# Create Works Enriched

## Overview

This notebook creates the final `openalex_works` table by enriching `openalex_works_base` with additional data from various sources. It deep clones the base table into `openalex_works_build`, merges the enrichments into that build, and publishes the finished build as `openalex_works` in one commit at the end (oxjob #2004), so a reader never sees a half-built `openalex_works`.

## Data Flow

```
openalex_works_base (from CreateWorksBase)
    |
    v [DEEP CLONE]
openalex_works_build
    |
    v [MERGE operations]
openalex_works_build (enriched)
    |
    v [publish, one commit: INSERT OVERWRITE with the stored dates (#679, #1873)]
openalex_works
```

## Enrichment Sources

### Referenced Works (combined from two sources)
1. **Legacy backfill:** `openalex.mid.citation` - historical citation data from MAG
2. **Parsed references:** `openalex.works.referenced_works` - resolved from raw references by `parse_work_references.ipynb`

Both are pre-unioned and deduped via `COLLECT_SET` in a single MERGE.

### Citations
- `cited_by_count` and `counts_by_year` - computed from `referenced_works` edges
- `fwci` and `citation_normalized_percentile` - computed from pub+3 citations within cohort

### Content
- **Fulltext:** from `openalex.works.work_fulltext` (built incrementally by `CreateWorkFulltext`); merged last
- **Concepts:** backfill + predicted (using `concept_key` hash)
- **Keywords:** `openalex.works.work_keywords_v2` (student tagger, oxjob #1322); the concept tagger's `work_concepts.keywords` is no longer served
- **Topics:** `openalex.works.work_topics_v2` (q8b_2m student, oxjob #1531): top 3, score = the model's probability that the topic is the work's main one; no topic for works the model calls not classifiable. The old `work_topics` (BERT + backfill) is no longer read and is kept for rollback
- **SDG:** `openalex.works.works_sdg_jev`, the Jev-trained head on the Qwen3 vectors (oxjob #1300; replaced Aurora's backfill + frontfill). Aurora's frozen tags were served as the deprecated `sustainable_development_goals_aurora` until 2026-11-02, then removed

### Metadata
- **Awards:** from `openalex.awards.work_awards`
- **Funders:** combined from backfill, fulltext extraction, and GTR
- **Authorships:** enriched with institution matching from `work_authorships`
- **Related works:** from `openalex.works.work_related_works` (oxjob #1648): the 10 nearest works by the Qwen3 vector (title, abstract, venue) under policy v1.2. The old concept-based one-time copy `related_works_backfill` (2025-09-15) is no longer read and is kept for rollback
- **Work type:** no longer merged here — set in CreateWorksBase from the primary location's type

### Build and enrich `openalex.works.openalex_works_build`; the last cell publishes it as `openalex.works.openalex_works`

## `openalex_works_hash` is persistent (oxjob #679)

This table — `(id, updated_date, content_hash)` — is the **authoritative owner of
`updated_date`**. It is never `CREATE OR REPLACE`'d by the pipeline: a work's
`updated_date` changes **only** when its `content_hash` changes (maintenance
MERGE near the end of this notebook), so it survives the nightly `DEEP CLONE`
of `openalex_works_base` into `openalex_works_build`. Before this change, a pre-run snapshot was recreated here
every night and enrichment bumps lived for exactly one run — any bump not
consumed by a sync before the next clone was silently lost (#679).

Consumers keep reading `openalex_works.updated_date` as normal: the publish cell
(last) serves the dates from this table every run, replacing whatever the clone
reverted (#2004). The column is therefore identical to the hash table (modulo
NULL-date rows) at the end of every run.

**One-time bootstrap** (already run; repeat only if the table is lost) — seed
from the current post-run state (`qa/oxjob679-bootstrap-works-hash.sql`):

```sql
CREATE OR REPLACE TABLE openalex.works.openalex_works_hash AS
SELECT id, updated_date, <content-hash expression from the cell below> AS content_hash
FROM openalex.works.openalex_works;
```

In [ ]:
-- oxjob #2004: build aside. This cell and every MERGE below write openalex_works_build; the last cell of the notebook
-- publishes the finished build as openalex_works in one commit. Until then every reader, inside the End 2 End or not,
-- reads the previous finished build. 7-day retention: the build carries the MERGEs' file churn, and predictive
-- optimization vacuums it by this setting.
CREATE OR REPLACE TABLE openalex.works.openalex_works_build
DEEP CLONE openalex.works.openalex_works_base
TBLPROPERTIES (
  'delta.dataSkippingNumIndexedCols' = 36,
  'delta.deletedFileRetentionDuration' = '7 days',
  'delta.logRetentionDuration' = '7 days'
);

### Work types v2: the decision beside `type`, and `type` where the switch serves (oxjob #1976)

In [ ]:
-- Work types v2 (oxjob #1976): the decision beside type, and type itself only where the switch serves.
-- GENERATED by notebooks/work_types/lib/work_type_cell.py from the shipped policies (pol-9b8e9d427e64, pol4-84165a93257e, pol5-4ed2aed64c8e, pol5-b007d7259869);
-- edit the generator or the switch file, never this cell. Switch (lib/work_type_switch.json): wt_policy_id pol5-4ed2aed64c8e, wt_served rules, wt_n_buckets 4, wt_start_buckets 0, wt_step 1, wt_switch_date 2026-11-01.
-- Reads work_type_scores (our job writes it after
-- each End 2 End), tonight's build (the cascade's type), last night's openalex_works (to explain flips) and
-- common.work_types; writes openalex_works_build only. Any error: the handler keeps last night's type_decision and
-- served types, End 2 End publishes as usual, and Classify_Work_Types' Check_Flips fails loud the same day. Runs before
-- FWCI (cell 12) and the content hash (cell 26), which read type; type_decision is not hashed, so the shadow alone
-- re-stamps nothing.
BEGIN
  DECLARE EXIT HANDLER FOR SQLEXCEPTION
  BEGIN
    DECLARE EXIT HANDLER FOR SQLEXCEPTION
      SELECT 'work types: the decision and its fallback both failed; no type_decision tonight' AS work_types;
    IF NOT (SELECT count(*) FROM openalex.information_schema.columns WHERE table_schema = 'works' AND table_name = 'openalex_works_build' AND column_name = 'type_decision') > 0 THEN
      ALTER TABLE openalex.works.openalex_works_build ADD COLUMNS (type_decision STRUCT<type: STRING, outcome: STRING, decided_by: STRING, confidence: DECIMAL(5,4), input_hash: STRING, s2_type: STRING, s2_confidence: DECIMAL(5,4), s2_input_hash: STRING, today_type: STRING, policy_id: STRING, model_version: STRING, vocab_version: STRING, is_traditional: BOOLEAN, flip: STRING, served: BOOLEAN, held: BOOLEAN, held_type: STRING, decided_on: DATE>);
    END IF;
    IF (SELECT count(*) FROM openalex.information_schema.columns WHERE table_schema = 'works' AND table_name = 'openalex_works' AND column_name = 'type_decision') > 0 THEN
      MERGE INTO openalex.works.openalex_works_build AS t
      USING (SELECT id, type, type_decision FROM openalex.works.openalex_works WHERE type_decision IS NOT NULL) AS p
      ON t.id = p.id
      WHEN MATCHED THEN UPDATE SET t.type_decision = p.type_decision,
        t.type = CASE WHEN p.type_decision.served THEN p.type ELSE t.type END;
    END IF;
    SELECT 'work types: the decision failed; last night''s decisions and served types kept' AS work_types;
  END;
  -- the switch, checked in a statement of its own (one row, always evaluated): a bad value fails here, into the handler
  WITH prm AS (SELECT CAST('pol5-4ed2aed64c8e' AS STRING) AS policy_id, CAST('rules' AS STRING) AS served, CAST('4' AS INT) AS n_buckets, CAST('0' AS INT) AS start_buckets, CAST('1' AS INT) AS step, CAST('2026-11-01' AS DATE) AS switch_date),
  pol AS (SELECT p.* FROM (SELECT 'pol-9b8e9d427e64' AS policy_id, 's2-de2ab7cf26b0f7a0' AS model_id, CAST(0.7000 AS DECIMAL(5,4)) AS tau, array('article', 'book', 'book-chapter', 'book-review', 'conference-abstract', 'conference-paper', 'data-paper', 'dataset', 'dissertation', 'editorial', 'erratum', 'letter', 'libguides', 'other', 'paratext', 'peer-review', 'preprint', 'reference-entry', 'report', 'retraction', 'review', 'software-paper', 'software', 'standard', 'supplementary-materials', 'study-registration') AS classes, 'C4' AS policy_kind UNION ALL SELECT 'pol4-84165a93257e' AS policy_id, 's2-0bc40dd7f812bdfe' AS model_id, CAST(0.8000 AS DECIMAL(5,4)) AS tau, array('article', 'book', 'book-chapter', 'book-review', 'conference-abstract', 'conference-paper', 'data-paper', 'dataset', 'dissertation', 'editorial', 'erratum', 'letter', 'other', 'paratext', 'peer-review', 'preprint', 'reference-entry', 'report', 'retraction', 'review', 'software-paper', 'software', 'standard', 'supplementary-materials', 'study-registration') AS classes, 'C4+stage0' AS policy_kind UNION ALL SELECT 'pol5-4ed2aed64c8e' AS policy_id, 's2-0bc40dd7f812bdfe' AS model_id, CAST(0.8000 AS DECIMAL(5,4)) AS tau, array('article', 'book', 'book-chapter', 'book-review', 'conference-abstract', 'conference-paper', 'data-paper', 'dataset', 'dissertation', 'editorial', 'erratum', 'letter', 'other', 'paratext', 'peer-review', 'preprint', 'reference-entry', 'report', 'retraction', 'review', 'software-paper', 'software', 'standard', 'supplementary-materials', 'study-registration') AS classes, 'C4+cuts+stage0' AS policy_kind UNION ALL SELECT 'pol5-b007d7259869' AS policy_id, 's2-0bc40dd7f812bdfe' AS model_id, CAST(0.8000 AS DECIMAL(5,4)) AS tau, array('article', 'book', 'book-chapter', 'book-review', 'conference-abstract', 'conference-paper', 'data-paper', 'dataset', 'dissertation', 'editorial', 'erratum', 'letter', 'other', 'paratext', 'peer-review', 'preprint', 'reference-entry', 'report', 'retraction', 'review', 'software-paper', 'software', 'standard', 'supplementary-materials', 'study-registration') AS classes, 'C4+cuts+stage0' AS policy_kind) p JOIN prm ON p.policy_id = prm.policy_id)
  SELECT CASE WHEN (SELECT count(*) FROM pol) = 1 AND prm.served IN ('rules', prm.policy_id) AND prm.n_buckets >= 1
                   AND prm.start_buckets BETWEEN 0 AND prm.n_buckets AND prm.step IN (-1, 1)
              THEN 'work types: switch ok' ELSE raise_error(concat('work types: unknown policy or bad switch parameters: ', prm.policy_id, ' / ', prm.served)) END AS work_types
  FROM prm;
    IF NOT (SELECT count(*) FROM openalex.information_schema.columns WHERE table_schema = 'works' AND table_name = 'openalex_works_build' AND column_name = 'type_decision') > 0 THEN
      ALTER TABLE openalex.works.openalex_works_build ADD COLUMNS (type_decision STRUCT<type: STRING, outcome: STRING, decided_by: STRING, confidence: DECIMAL(5,4), input_hash: STRING, s2_type: STRING, s2_confidence: DECIMAL(5,4), s2_input_hash: STRING, today_type: STRING, policy_id: STRING, model_version: STRING, vocab_version: STRING, is_traditional: BOOLEAN, flip: STRING, served: BOOLEAN, held: BOOLEAN, held_type: STRING, decided_on: DATE>);
    END IF;
  IF (SELECT count(*) FROM openalex.information_schema.columns WHERE table_schema = 'works' AND table_name = 'openalex_works' AND column_name = 'type_decision') > 0 THEN
    MERGE INTO openalex.works.openalex_works_build AS t
    USING (
WITH prm AS (SELECT CAST('pol5-4ed2aed64c8e' AS STRING) AS policy_id, CAST('rules' AS STRING) AS served, CAST('4' AS INT) AS n_buckets, CAST('0' AS INT) AS start_buckets, CAST('1' AS INT) AS step, CAST('2026-11-01' AS DATE) AS switch_date),
pol AS (SELECT p.* FROM (SELECT 'pol-9b8e9d427e64' AS policy_id, 's2-de2ab7cf26b0f7a0' AS model_id, CAST(0.7000 AS DECIMAL(5,4)) AS tau, array('article', 'book', 'book-chapter', 'book-review', 'conference-abstract', 'conference-paper', 'data-paper', 'dataset', 'dissertation', 'editorial', 'erratum', 'letter', 'libguides', 'other', 'paratext', 'peer-review', 'preprint', 'reference-entry', 'report', 'retraction', 'review', 'software-paper', 'software', 'standard', 'supplementary-materials', 'study-registration') AS classes, 'C4' AS policy_kind UNION ALL SELECT 'pol4-84165a93257e' AS policy_id, 's2-0bc40dd7f812bdfe' AS model_id, CAST(0.8000 AS DECIMAL(5,4)) AS tau, array('article', 'book', 'book-chapter', 'book-review', 'conference-abstract', 'conference-paper', 'data-paper', 'dataset', 'dissertation', 'editorial', 'erratum', 'letter', 'other', 'paratext', 'peer-review', 'preprint', 'reference-entry', 'report', 'retraction', 'review', 'software-paper', 'software', 'standard', 'supplementary-materials', 'study-registration') AS classes, 'C4+stage0' AS policy_kind UNION ALL SELECT 'pol5-4ed2aed64c8e' AS policy_id, 's2-0bc40dd7f812bdfe' AS model_id, CAST(0.8000 AS DECIMAL(5,4)) AS tau, array('article', 'book', 'book-chapter', 'book-review', 'conference-abstract', 'conference-paper', 'data-paper', 'dataset', 'dissertation', 'editorial', 'erratum', 'letter', 'other', 'paratext', 'peer-review', 'preprint', 'reference-entry', 'report', 'retraction', 'review', 'software-paper', 'software', 'standard', 'supplementary-materials', 'study-registration') AS classes, 'C4+cuts+stage0' AS policy_kind UNION ALL SELECT 'pol5-b007d7259869' AS policy_id, 's2-0bc40dd7f812bdfe' AS model_id, CAST(0.8000 AS DECIMAL(5,4)) AS tau, array('article', 'book', 'book-chapter', 'book-review', 'conference-abstract', 'conference-paper', 'data-paper', 'dataset', 'dissertation', 'editorial', 'erratum', 'letter', 'other', 'paratext', 'peer-review', 'preprint', 'reference-entry', 'report', 'retraction', 'review', 'software-paper', 'software', 'standard', 'supplementary-materials', 'study-registration') AS classes, 'C4+cuts+stage0' AS policy_kind) p JOIN prm ON p.policy_id = prm.policy_id),
-- the switch (rule 6): buckets served tonight = clamp(start + step * days since switch_date, 0, N); a bad parameter or an
-- unknown policy fails the statement, and the cell's handler keeps last night's decisions and types
sw AS (
  SELECT prm.served, prm.n_buckets,
         CASE WHEN (SELECT count(*) FROM pol) = 1 AND prm.served IN ('rules', prm.policy_id) AND prm.n_buckets >= 1
                   AND prm.start_buckets BETWEEN 0 AND prm.n_buckets AND prm.step IN (-1, 1)
              THEN greatest(0, least(prm.n_buckets, prm.start_buckets + prm.step * datediff(current_date(), prm.switch_date)))
              ELSE raise_error(concat('work types: unknown policy or bad switch parameters: ', prm.policy_id, ' / ', prm.served)) END AS k
  FROM prm
),
cuts AS (SELECT c.type, c.cut FROM (SELECT 'pol5-4ed2aed64c8e' AS policy_id, 'reference-entry' AS type, CAST(0.9914 AS DECIMAL(5,4)) AS cut UNION ALL SELECT 'pol5-b007d7259869' AS policy_id, 'conference-abstract' AS type, CAST(NULL AS DECIMAL(5,4)) AS cut UNION ALL SELECT 'pol5-b007d7259869' AS policy_id, 'reference-entry' AS type, CAST(0.9914 AS DECIMAL(5,4)) AS cut) c JOIN prm ON c.policy_id = prm.policy_id),
lists AS (SELECT l.* FROM (SELECT 'pol-9b8e9d427e64' AS policy_id, x.* FROM (SELECT CAST(NULL AS INT) AS ord, CAST(NULL AS STRING) AS list_id, CAST(NULL AS STRING) AS type, CAST(array() AS ARRAY<STRING>) AS doi_prefixes, CAST(NULL AS STRING) AS doi_regex, CAST(array() AS ARRAY<STRING>) AS names_exact, CAST(NULL AS STRING) AS names_regex, CAST(array() AS ARRAY<STRING>) AS url_substrings WHERE 1 = 0) x UNION ALL SELECT 'pol4-84165a93257e' AS policy_id, x.* FROM (SELECT 0 AS ord, 's0-oxford_reference_works-6d562bd9' AS list_id, 'reference-entry' AS type, CAST(array() AS ARRAY<STRING>) AS doi_prefixes, '^10\\.1093/(acrefore|gao|oao|acref)/' AS doi_regex, CAST(array() AS ARRAY<STRING>) AS names_exact, CAST(NULL AS STRING) AS names_regex, CAST(array() AS ARRAY<STRING>) AS url_substrings UNION ALL SELECT 1 AS ord, 's0-specimen_sample_registries_v6-536c5a74' AS list_id, 'other' AS type, array('10.3535', '10.18730') AS doi_prefixes, CAST(NULL AS STRING) AS doi_regex, array('distributed system of scientific collections', 'dissco', 'international treaty on plant genetic resources for food and agriculture') AS names_exact, '^sesar(_|$|\\b)|\\bigsn\\b' AS names_regex, CAST(array() AS ARRAY<STRING>) AS url_substrings UNION ALL SELECT 2 AS ord, 's0-wiley_mrw-a52c6ceb' AS list_id, 'reference-entry' AS type, CAST(array() AS ARRAY<STRING>) AS doi_prefixes, '^10\\.1002/(9780470015902\\.a|0471238961\\.|14356007\\.|9780470027318\\.a|9781118445112\\.stat)' AS doi_regex, CAST(array() AS ARRAY<STRING>) AS names_exact, CAST(NULL AS STRING) AS names_regex, CAST(array() AS ARRAY<STRING>) AS url_substrings) x UNION ALL SELECT 'pol5-4ed2aed64c8e' AS policy_id, x.* FROM (SELECT 0 AS ord, 's0-oxford_reference_works-6d562bd9' AS list_id, 'reference-entry' AS type, CAST(array() AS ARRAY<STRING>) AS doi_prefixes, '^10\\.1093/(acrefore|gao|oao|acref)/' AS doi_regex, CAST(array() AS ARRAY<STRING>) AS names_exact, CAST(NULL AS STRING) AS names_regex, CAST(array() AS ARRAY<STRING>) AS url_substrings UNION ALL SELECT 1 AS ord, 's0-specimen_sample_registries_v6-536c5a74' AS list_id, 'other' AS type, array('10.3535', '10.18730') AS doi_prefixes, CAST(NULL AS STRING) AS doi_regex, array('distributed system of scientific collections', 'dissco', 'international treaty on plant genetic resources for food and agriculture') AS names_exact, '^sesar(_|$|\\b)|\\bigsn\\b' AS names_regex, CAST(array() AS ARRAY<STRING>) AS url_substrings UNION ALL SELECT 2 AS ord, 's0-wiley_mrw-a52c6ceb' AS list_id, 'reference-entry' AS type, CAST(array() AS ARRAY<STRING>) AS doi_prefixes, '^10\\.1002/(9780470015902\\.a|0471238961\\.|14356007\\.|9780470027318\\.a|9781118445112\\.stat)' AS doi_regex, CAST(array() AS ARRAY<STRING>) AS names_exact, CAST(NULL AS STRING) AS names_regex, CAST(array() AS ARRAY<STRING>) AS url_substrings) x UNION ALL SELECT 'pol5-b007d7259869' AS policy_id, x.* FROM (SELECT 0 AS ord, 's0-oxford_reference_works-6d562bd9' AS list_id, 'reference-entry' AS type, CAST(array() AS ARRAY<STRING>) AS doi_prefixes, '^10\\.1093/(acrefore|gao|oao|acref)/' AS doi_regex, CAST(array() AS ARRAY<STRING>) AS names_exact, CAST(NULL AS STRING) AS names_regex, CAST(array() AS ARRAY<STRING>) AS url_substrings UNION ALL SELECT 1 AS ord, 's0-specimen_sample_registries_v6-536c5a74' AS list_id, 'other' AS type, array('10.3535', '10.18730') AS doi_prefixes, CAST(NULL AS STRING) AS doi_regex, array('distributed system of scientific collections', 'dissco', 'international treaty on plant genetic resources for food and agriculture') AS names_exact, '^sesar(_|$|\\b)|\\bigsn\\b' AS names_regex, CAST(array() AS ARRAY<STRING>) AS url_substrings UNION ALL SELECT 2 AS ord, 's0-wiley_mrw-a52c6ceb' AS list_id, 'reference-entry' AS type, CAST(array() AS ARRAY<STRING>) AS doi_prefixes, '^10\\.1002/(9780470015902\\.a|0471238961\\.|14356007\\.|9780470027318\\.a|9781118445112\\.stat)' AS doi_regex, CAST(array() AS ARRAY<STRING>) AS names_exact, CAST(NULL AS STRING) AS names_regex, CAST(array() AS ARRAY<STRING>) AS url_substrings) x) l JOIN prm ON l.policy_id = prm.policy_id),
ty AS (SELECT id AS type, is_traditional FROM openalex.common.work_types),
vv AS (SELECT concat('vocab-', substr(sha2(concat('[', concat_ws(',', array_sort(collect_list(concat('["', type, '",', CASE WHEN is_traditional THEN 'true' ELSE 'false' END, ']')))), ']'), 256), 1, 12)) AS vocab_version FROM ty),
-- each work's current score: the latest-seen row of the policy's model (the queue builder touches a row when the
-- work's inputs come back to an already-scored text), with the pack's stage-0 signals
sc AS (
  SELECT s.work_id, s.input_hash AS s2_input_hash, s.class_idx, s.confidence AS conf_stored,
         coalesce(s.s0_doi, '') AS s0_doi, s.s0_names, s.s0_urls
  FROM openalex.works.work_type_scores s JOIN pol ON s.model_version = pol.model_id
  QUALIFY row_number() OVER (PARTITION BY s.work_id ORDER BY s.seen_at DESC, s.scored_at DESC, s.input_hash) = 1
),
-- tonight's cascade type (CreateWorksBase), normalised as vocab.norm_type; non-xpac works only (include_xpac is off)
bw AS (
  SELECT id AS work_id, type AS build_type, primary_location.provenance AS pl_prov, primary_location.native_id AS pl_nid,
         lower(regexp_replace(regexp_replace(type, '^\\s+', ''), '\\s+$', '')) AS x
  FROM openalex.works.openalex_works_build
  WHERE NOT coalesce(is_xpac, false)
),
j AS (
  SELECT bw.work_id, bw.build_type, bw.pl_prov, bw.pl_nid,
         CASE WHEN bw.x IS NULL OR bw.x = '' THEN NULL WHEN bw.x = 'libguide' THEN 'libguides'
              WHEN bw.x = 'conference-proceedings' THEN 'conference-paper' WHEN bw.x = 'thesis' THEN 'dissertation'
              ELSE bw.x END AS today_type,
         sc.s2_input_hash, sc.class_idx, sc.conf_stored, sc.s0_doi, sc.s0_names, sc.s0_urls
  FROM bw JOIN sc ON sc.work_id = bw.work_id
),
dd AS (
  SELECT j.*, pol.policy_id, pol.model_id, pol.tau,
         CASE WHEN j.class_idx BETWEEN 0 AND size(pol.classes) - 1 THEN element_at(pol.classes, j.class_idx + 1)
              ELSE raise_error(concat('class_idx outside the model class list: work ', CAST(j.work_id AS STRING))) END AS s2_type,
         CASE WHEN CAST(j.conf_stored AS DECIMAL(5,4)) = j.conf_stored THEN CAST(j.conf_stored AS DECIMAL(5,4))
              ELSE raise_error(concat('confidence is not a stored 4-dp value: work ', CAST(j.work_id AS STRING))) END AS s2_confidence
  FROM j CROSS JOIN pol
),
-- C4, fallback today (student/prod writer_lists.write_rows_cuts_lists): stage 2's type where its stored 4-dp confidence
-- passes tau and the type's cut (a null cut = never), and the type is live; else today's type if live
pp AS (
  SELECT dd.*, ts.type IS NOT NULL AND dd.s2_confidence >= dd.tau
               AND (c.type IS NULL OR (c.cut IS NOT NULL AND dd.s2_confidence >= c.cut)) AS passes,
         ts.is_traditional AS s2_trad, tt.type IS NOT NULL AS today_live, tt.is_traditional AS today_trad
  FROM dd
  LEFT JOIN ty ts ON ts.type = dd.s2_type
  LEFT JOIN ty tt ON tt.type = dd.today_type
  LEFT JOIN cuts c ON c.type = dd.s2_type
),
b AS (
  SELECT work_id, build_type, pl_prov, pl_nid, s0_doi, s0_names, s0_urls, policy_id, model_id,
         CASE WHEN passes THEN s2_type WHEN today_live THEN today_type END AS type_v2,
         CASE WHEN passes THEN s2_confidence END AS confidence,
         CASE WHEN passes THEN concat('student-s2:', model_id) WHEN today_type IS NULL THEN 'no_today_type' ELSE 'fallback:today' END AS decided_by,
         CASE WHEN passes THEN 'student-s2' WHEN today_type IS NULL THEN 'no_today_type' ELSE 'fallback:today' END AS outcome,
         CASE WHEN passes THEN s2_trad WHEN today_live THEN today_trad END AS is_traditional,
         substr(sha2(concat_ws('|', '', s2_input_hash, coalesce(today_type, ''), '', ''), 256), 1, 32) AS input_hash,
         s2_type, s2_confidence, s2_input_hash, today_type
  FROM pp
),
-- stage-0 source lists (student/prod stage0.apply): a work matched by lists of exactly one type is settled by the first
-- such list in the policy's order; a list with both a DOI regex and URL substrings needs both to hit
l1 AS (SELECT * FROM lists WHERE NOT (doi_regex IS NOT NULL AND size(url_substrings) > 0)),
l2 AS (SELECT * FROM lists WHERE doi_regex IS NOT NULL AND size(url_substrings) > 0),
s0m AS (
  SELECT b.work_id, l1.ord, l1.list_id, l1.type FROM b CROSS JOIN l1
  WHERE (instr(b.s0_doi, '/') > 0 AND array_contains(l1.doi_prefixes, split_part(b.s0_doi, '/', 1)))
     OR (l1.doi_regex IS NOT NULL AND b.s0_doi RLIKE l1.doi_regex)
  UNION ALL
  SELECT x.work_id, l1.ord, l1.list_id, l1.type
  FROM (SELECT work_id, explode(s0_names) AS n FROM b WHERE s0_names IS NOT NULL) x CROSS JOIN l1
  WHERE (size(l1.names_exact) > 0 OR l1.names_regex IS NOT NULL)
    AND (array_contains(l1.names_exact, x.n) OR (l1.names_regex IS NOT NULL AND x.n RLIKE l1.names_regex))
  UNION ALL
  SELECT x.work_id, s.ord, s.list_id, s.type
  FROM (SELECT work_id, s0_doi AS doi, explode(s0_urls) AS u FROM b WHERE s0_urls IS NOT NULL) x
  CROSS JOIN (SELECT ord, list_id, type, doi_regex, explode(url_substrings) AS sub FROM l2) s
  WHERE x.doi RLIKE s.doi_regex AND instr(x.u, s.sub) > 0
),
s0 AS (
  SELECT work_id, min_by(list_id, ord) AS list_id, min(type) AS s0_type
  FROM (SELECT DISTINCT work_id, ord, list_id, type FROM s0m) m
  GROUP BY work_id
  HAVING count(DISTINCT type) = 1
),
dec AS (
  SELECT b.work_id, b.build_type, b.pl_prov, b.pl_nid, b.policy_id, b.model_id,
         CASE WHEN t0.type IS NOT NULL THEN s0.s0_type ELSE b.type_v2 END AS type_v2,
         CASE WHEN t0.type IS NOT NULL THEN NULL ELSE b.confidence END AS confidence,
         CASE WHEN t0.type IS NOT NULL THEN concat('stage0:', s0.list_id) ELSE b.decided_by END AS decided_by,
         CASE WHEN t0.type IS NOT NULL THEN 'stage0-list' ELSE b.outcome END AS outcome,
         CASE WHEN t0.type IS NOT NULL THEN t0.is_traditional ELSE b.is_traditional END AS is_traditional,
         CASE WHEN t0.type IS NOT NULL THEN substr(sha2(concat_ws('|', b.input_hash, s0.list_id), 256), 1, 32) ELSE b.input_hash END AS input_hash,
         b.s2_type, b.s2_confidence, b.s2_input_hash, b.today_type
  FROM b
  LEFT JOIN s0 ON s0.work_id = b.work_id
  LEFT JOIN ty t0 ON t0.type = s0.s0_type                      -- a list type retired since the policy was made never settles
),
prot AS (SELECT DISTINCT provenance, native_id FROM openalex.works.locations_w_types WHERE classified_rule IN ('#1647: physicalobject sample code ahead of title code -> other')),
del AS (SELECT DISTINCT work_id FROM openalex.works.deleted_works),
prev AS (SELECT id AS work_id, type AS pub_type, type_decision AS pv FROM openalex.works.openalex_works),
-- why the decision differs from last night's published one (null: it doesn't); 'unexplained' = nothing it rests on moved
f AS (
  SELECT dec.*, prev.pub_type, vv.vocab_version, prev.pv.type AS p_type, prev.pv.outcome AS p_outcome,
         prev.pv.decided_by AS p_decided_by, prev.pv.confidence AS p_confidence, prev.pv.is_traditional AS p_trad,
         CASE WHEN prev.pv IS NULL THEN 'new'
              WHEN prev.pv.type IS NOT DISTINCT FROM dec.type_v2 THEN NULL
              WHEN prev.pv.policy_id IS DISTINCT FROM dec.policy_id THEN 'policy'
              WHEN prev.pv.model_version IS DISTINCT FROM dec.model_id THEN 'model'
              WHEN prev.pv.vocab_version IS DISTINCT FROM vv.vocab_version THEN 'vocabulary'
              WHEN prev.pv.s2_input_hash IS DISTINCT FROM dec.s2_input_hash THEN 'input'
              WHEN (prev.pv.outcome = 'stage0-list' OR dec.outcome = 'stage0-list')
                   AND prev.pv.decided_by IS DISTINCT FROM dec.decided_by THEN 'stage0-list'
              WHEN dec.outcome <> 'student-s2' AND prev.pv.today_type IS DISTINCT FROM dec.today_type THEN 'today_type'
              ELSE 'unexplained' END AS flip,
         pr.native_id IS NOT NULL AS protected, dl.work_id IS NOT NULL AS deleted
  FROM dec CROSS JOIN vv
  LEFT JOIN prev ON prev.work_id = dec.work_id
  LEFT JOIN prot pr ON pr.provenance = dec.pl_prov AND pr.native_id = dec.pl_nid
  LEFT JOIN del dl ON dl.work_id = dec.work_id
),
-- an unexplained flip holds last night's decision (type, outcome, ...) until something it rests on moves: the next
-- night compares against the held decision, so the flip stays 'unexplained' (and Check_Flips loud) every night; the
-- new decision waits in held_type (Codex 11-09 #2)
h AS (
  SELECT f.*, flip = 'unexplained' AS held,
         CASE WHEN flip = 'unexplained' THEN p_type ELSE type_v2 END AS d_type,
         CASE WHEN flip = 'unexplained' THEN p_outcome ELSE outcome END AS d_outcome,
         CASE WHEN flip = 'unexplained' THEN p_decided_by ELSE decided_by END AS d_decided_by,
         CASE WHEN flip = 'unexplained' THEN p_confidence ELSE confidence END AS d_confidence,
         CASE WHEN flip = 'unexplained' THEN p_trad ELSE is_traditional END AS d_trad,
         CASE WHEN flip = 'unexplained' THEN type_v2 END AS held_type
  FROM f
),
-- served tonight: rule 1 stage-2 and list decisions only (fallback works keep tonight's cascade type); 2 live types only
-- (both kinds are live by construction); 3 never a removed work; 4 no study-registration written, and Jason's
-- study-registration rows stay; 4b protected cascade rules stay; 5 parent/child (#2065, not live yet); 6 the ramp
sv AS (
  SELECT h.*,
         sw.served = h.policy_id AND h.d_outcome IN ('student-s2', 'stage0-list') AND h.d_type IS NOT NULL
         AND h.d_type <> 'study-registration' AND coalesce(h.build_type, '') <> 'study-registration'
         AND NOT h.protected AND NOT h.deleted
         AND pmod(xxhash64(h.work_id), sw.n_buckets) < sw.k AS serve
  FROM h CROSS JOIN sw
)
SELECT work_id, d_type AS serve_type, CAST(named_struct('type', d_type, 'outcome', d_outcome, 'decided_by', d_decided_by, 'confidence', d_confidence, 'input_hash', input_hash, 's2_type', s2_type, 's2_confidence', s2_confidence, 's2_input_hash', s2_input_hash, 'today_type', today_type, 'policy_id', policy_id, 'model_version', model_id, 'vocab_version', vocab_version, 'is_traditional', d_trad, 'flip', flip, 'served', serve, 'held', held, 'held_type', held_type, 'decided_on', current_date()) AS STRUCT<type: STRING, outcome: STRING, decided_by: STRING, confidence: DECIMAL(5,4), input_hash: STRING, s2_type: STRING, s2_confidence: DECIMAL(5,4), s2_input_hash: STRING, today_type: STRING, policy_id: STRING, model_version: STRING, vocab_version: STRING, is_traditional: BOOLEAN, flip: STRING, served: BOOLEAN, held: BOOLEAN, held_type: STRING, decided_on: DATE>) AS v2
FROM sv
    ) AS s
    ON t.id = s.work_id
    WHEN MATCHED THEN UPDATE SET t.type_decision = s.v2,
      t.type = CASE WHEN s.v2.served THEN s.serve_type ELSE t.type END;
  ELSE
    -- the first night: nothing published to compare with yet
    MERGE INTO openalex.works.openalex_works_build AS t
    USING (
WITH prm AS (SELECT CAST('pol5-4ed2aed64c8e' AS STRING) AS policy_id, CAST('rules' AS STRING) AS served, CAST('4' AS INT) AS n_buckets, CAST('0' AS INT) AS start_buckets, CAST('1' AS INT) AS step, CAST('2026-11-01' AS DATE) AS switch_date),
pol AS (SELECT p.* FROM (SELECT 'pol-9b8e9d427e64' AS policy_id, 's2-de2ab7cf26b0f7a0' AS model_id, CAST(0.7000 AS DECIMAL(5,4)) AS tau, array('article', 'book', 'book-chapter', 'book-review', 'conference-abstract', 'conference-paper', 'data-paper', 'dataset', 'dissertation', 'editorial', 'erratum', 'letter', 'libguides', 'other', 'paratext', 'peer-review', 'preprint', 'reference-entry', 'report', 'retraction', 'review', 'software-paper', 'software', 'standard', 'supplementary-materials', 'study-registration') AS classes, 'C4' AS policy_kind UNION ALL SELECT 'pol4-84165a93257e' AS policy_id, 's2-0bc40dd7f812bdfe' AS model_id, CAST(0.8000 AS DECIMAL(5,4)) AS tau, array('article', 'book', 'book-chapter', 'book-review', 'conference-abstract', 'conference-paper', 'data-paper', 'dataset', 'dissertation', 'editorial', 'erratum', 'letter', 'other', 'paratext', 'peer-review', 'preprint', 'reference-entry', 'report', 'retraction', 'review', 'software-paper', 'software', 'standard', 'supplementary-materials', 'study-registration') AS classes, 'C4+stage0' AS policy_kind UNION ALL SELECT 'pol5-4ed2aed64c8e' AS policy_id, 's2-0bc40dd7f812bdfe' AS model_id, CAST(0.8000 AS DECIMAL(5,4)) AS tau, array('article', 'book', 'book-chapter', 'book-review', 'conference-abstract', 'conference-paper', 'data-paper', 'dataset', 'dissertation', 'editorial', 'erratum', 'letter', 'other', 'paratext', 'peer-review', 'preprint', 'reference-entry', 'report', 'retraction', 'review', 'software-paper', 'software', 'standard', 'supplementary-materials', 'study-registration') AS classes, 'C4+cuts+stage0' AS policy_kind UNION ALL SELECT 'pol5-b007d7259869' AS policy_id, 's2-0bc40dd7f812bdfe' AS model_id, CAST(0.8000 AS DECIMAL(5,4)) AS tau, array('article', 'book', 'book-chapter', 'book-review', 'conference-abstract', 'conference-paper', 'data-paper', 'dataset', 'dissertation', 'editorial', 'erratum', 'letter', 'other', 'paratext', 'peer-review', 'preprint', 'reference-entry', 'report', 'retraction', 'review', 'software-paper', 'software', 'standard', 'supplementary-materials', 'study-registration') AS classes, 'C4+cuts+stage0' AS policy_kind) p JOIN prm ON p.policy_id = prm.policy_id),
-- the switch (rule 6): buckets served tonight = clamp(start + step * days since switch_date, 0, N); a bad parameter or an
-- unknown policy fails the statement, and the cell's handler keeps last night's decisions and types
sw AS (
  SELECT prm.served, prm.n_buckets,
         CASE WHEN (SELECT count(*) FROM pol) = 1 AND prm.served IN ('rules', prm.policy_id) AND prm.n_buckets >= 1
                   AND prm.start_buckets BETWEEN 0 AND prm.n_buckets AND prm.step IN (-1, 1)
              THEN greatest(0, least(prm.n_buckets, prm.start_buckets + prm.step * datediff(current_date(), prm.switch_date)))
              ELSE raise_error(concat('work types: unknown policy or bad switch parameters: ', prm.policy_id, ' / ', prm.served)) END AS k
  FROM prm
),
cuts AS (SELECT c.type, c.cut FROM (SELECT 'pol5-4ed2aed64c8e' AS policy_id, 'reference-entry' AS type, CAST(0.9914 AS DECIMAL(5,4)) AS cut UNION ALL SELECT 'pol5-b007d7259869' AS policy_id, 'conference-abstract' AS type, CAST(NULL AS DECIMAL(5,4)) AS cut UNION ALL SELECT 'pol5-b007d7259869' AS policy_id, 'reference-entry' AS type, CAST(0.9914 AS DECIMAL(5,4)) AS cut) c JOIN prm ON c.policy_id = prm.policy_id),
lists AS (SELECT l.* FROM (SELECT 'pol-9b8e9d427e64' AS policy_id, x.* FROM (SELECT CAST(NULL AS INT) AS ord, CAST(NULL AS STRING) AS list_id, CAST(NULL AS STRING) AS type, CAST(array() AS ARRAY<STRING>) AS doi_prefixes, CAST(NULL AS STRING) AS doi_regex, CAST(array() AS ARRAY<STRING>) AS names_exact, CAST(NULL AS STRING) AS names_regex, CAST(array() AS ARRAY<STRING>) AS url_substrings WHERE 1 = 0) x UNION ALL SELECT 'pol4-84165a93257e' AS policy_id, x.* FROM (SELECT 0 AS ord, 's0-oxford_reference_works-6d562bd9' AS list_id, 'reference-entry' AS type, CAST(array() AS ARRAY<STRING>) AS doi_prefixes, '^10\\.1093/(acrefore|gao|oao|acref)/' AS doi_regex, CAST(array() AS ARRAY<STRING>) AS names_exact, CAST(NULL AS STRING) AS names_regex, CAST(array() AS ARRAY<STRING>) AS url_substrings UNION ALL SELECT 1 AS ord, 's0-specimen_sample_registries_v6-536c5a74' AS list_id, 'other' AS type, array('10.3535', '10.18730') AS doi_prefixes, CAST(NULL AS STRING) AS doi_regex, array('distributed system of scientific collections', 'dissco', 'international treaty on plant genetic resources for food and agriculture') AS names_exact, '^sesar(_|$|\\b)|\\bigsn\\b' AS names_regex, CAST(array() AS ARRAY<STRING>) AS url_substrings UNION ALL SELECT 2 AS ord, 's0-wiley_mrw-a52c6ceb' AS list_id, 'reference-entry' AS type, CAST(array() AS ARRAY<STRING>) AS doi_prefixes, '^10\\.1002/(9780470015902\\.a|0471238961\\.|14356007\\.|9780470027318\\.a|9781118445112\\.stat)' AS doi_regex, CAST(array() AS ARRAY<STRING>) AS names_exact, CAST(NULL AS STRING) AS names_regex, CAST(array() AS ARRAY<STRING>) AS url_substrings) x UNION ALL SELECT 'pol5-4ed2aed64c8e' AS policy_id, x.* FROM (SELECT 0 AS ord, 's0-oxford_reference_works-6d562bd9' AS list_id, 'reference-entry' AS type, CAST(array() AS ARRAY<STRING>) AS doi_prefixes, '^10\\.1093/(acrefore|gao|oao|acref)/' AS doi_regex, CAST(array() AS ARRAY<STRING>) AS names_exact, CAST(NULL AS STRING) AS names_regex, CAST(array() AS ARRAY<STRING>) AS url_substrings UNION ALL SELECT 1 AS ord, 's0-specimen_sample_registries_v6-536c5a74' AS list_id, 'other' AS type, array('10.3535', '10.18730') AS doi_prefixes, CAST(NULL AS STRING) AS doi_regex, array('distributed system of scientific collections', 'dissco', 'international treaty on plant genetic resources for food and agriculture') AS names_exact, '^sesar(_|$|\\b)|\\bigsn\\b' AS names_regex, CAST(array() AS ARRAY<STRING>) AS url_substrings UNION ALL SELECT 2 AS ord, 's0-wiley_mrw-a52c6ceb' AS list_id, 'reference-entry' AS type, CAST(array() AS ARRAY<STRING>) AS doi_prefixes, '^10\\.1002/(9780470015902\\.a|0471238961\\.|14356007\\.|9780470027318\\.a|9781118445112\\.stat)' AS doi_regex, CAST(array() AS ARRAY<STRING>) AS names_exact, CAST(NULL AS STRING) AS names_regex, CAST(array() AS ARRAY<STRING>) AS url_substrings) x UNION ALL SELECT 'pol5-b007d7259869' AS policy_id, x.* FROM (SELECT 0 AS ord, 's0-oxford_reference_works-6d562bd9' AS list_id, 'reference-entry' AS type, CAST(array() AS ARRAY<STRING>) AS doi_prefixes, '^10\\.1093/(acrefore|gao|oao|acref)/' AS doi_regex, CAST(array() AS ARRAY<STRING>) AS names_exact, CAST(NULL AS STRING) AS names_regex, CAST(array() AS ARRAY<STRING>) AS url_substrings UNION ALL SELECT 1 AS ord, 's0-specimen_sample_registries_v6-536c5a74' AS list_id, 'other' AS type, array('10.3535', '10.18730') AS doi_prefixes, CAST(NULL AS STRING) AS doi_regex, array('distributed system of scientific collections', 'dissco', 'international treaty on plant genetic resources for food and agriculture') AS names_exact, '^sesar(_|$|\\b)|\\bigsn\\b' AS names_regex, CAST(array() AS ARRAY<STRING>) AS url_substrings UNION ALL SELECT 2 AS ord, 's0-wiley_mrw-a52c6ceb' AS list_id, 'reference-entry' AS type, CAST(array() AS ARRAY<STRING>) AS doi_prefixes, '^10\\.1002/(9780470015902\\.a|0471238961\\.|14356007\\.|9780470027318\\.a|9781118445112\\.stat)' AS doi_regex, CAST(array() AS ARRAY<STRING>) AS names_exact, CAST(NULL AS STRING) AS names_regex, CAST(array() AS ARRAY<STRING>) AS url_substrings) x) l JOIN prm ON l.policy_id = prm.policy_id),
ty AS (SELECT id AS type, is_traditional FROM openalex.common.work_types),
vv AS (SELECT concat('vocab-', substr(sha2(concat('[', concat_ws(',', array_sort(collect_list(concat('["', type, '",', CASE WHEN is_traditional THEN 'true' ELSE 'false' END, ']')))), ']'), 256), 1, 12)) AS vocab_version FROM ty),
-- each work's current score: the latest-seen row of the policy's model (the queue builder touches a row when the
-- work's inputs come back to an already-scored text), with the pack's stage-0 signals
sc AS (
  SELECT s.work_id, s.input_hash AS s2_input_hash, s.class_idx, s.confidence AS conf_stored,
         coalesce(s.s0_doi, '') AS s0_doi, s.s0_names, s.s0_urls
  FROM openalex.works.work_type_scores s JOIN pol ON s.model_version = pol.model_id
  QUALIFY row_number() OVER (PARTITION BY s.work_id ORDER BY s.seen_at DESC, s.scored_at DESC, s.input_hash) = 1
),
-- tonight's cascade type (CreateWorksBase), normalised as vocab.norm_type; non-xpac works only (include_xpac is off)
bw AS (
  SELECT id AS work_id, type AS build_type, primary_location.provenance AS pl_prov, primary_location.native_id AS pl_nid,
         lower(regexp_replace(regexp_replace(type, '^\\s+', ''), '\\s+$', '')) AS x
  FROM openalex.works.openalex_works_build
  WHERE NOT coalesce(is_xpac, false)
),
j AS (
  SELECT bw.work_id, bw.build_type, bw.pl_prov, bw.pl_nid,
         CASE WHEN bw.x IS NULL OR bw.x = '' THEN NULL WHEN bw.x = 'libguide' THEN 'libguides'
              WHEN bw.x = 'conference-proceedings' THEN 'conference-paper' WHEN bw.x = 'thesis' THEN 'dissertation'
              ELSE bw.x END AS today_type,
         sc.s2_input_hash, sc.class_idx, sc.conf_stored, sc.s0_doi, sc.s0_names, sc.s0_urls
  FROM bw JOIN sc ON sc.work_id = bw.work_id
),
dd AS (
  SELECT j.*, pol.policy_id, pol.model_id, pol.tau,
         CASE WHEN j.class_idx BETWEEN 0 AND size(pol.classes) - 1 THEN element_at(pol.classes, j.class_idx + 1)
              ELSE raise_error(concat('class_idx outside the model class list: work ', CAST(j.work_id AS STRING))) END AS s2_type,
         CASE WHEN CAST(j.conf_stored AS DECIMAL(5,4)) = j.conf_stored THEN CAST(j.conf_stored AS DECIMAL(5,4))
              ELSE raise_error(concat('confidence is not a stored 4-dp value: work ', CAST(j.work_id AS STRING))) END AS s2_confidence
  FROM j CROSS JOIN pol
),
-- C4, fallback today (student/prod writer_lists.write_rows_cuts_lists): stage 2's type where its stored 4-dp confidence
-- passes tau and the type's cut (a null cut = never), and the type is live; else today's type if live
pp AS (
  SELECT dd.*, ts.type IS NOT NULL AND dd.s2_confidence >= dd.tau
               AND (c.type IS NULL OR (c.cut IS NOT NULL AND dd.s2_confidence >= c.cut)) AS passes,
         ts.is_traditional AS s2_trad, tt.type IS NOT NULL AS today_live, tt.is_traditional AS today_trad
  FROM dd
  LEFT JOIN ty ts ON ts.type = dd.s2_type
  LEFT JOIN ty tt ON tt.type = dd.today_type
  LEFT JOIN cuts c ON c.type = dd.s2_type
),
b AS (
  SELECT work_id, build_type, pl_prov, pl_nid, s0_doi, s0_names, s0_urls, policy_id, model_id,
         CASE WHEN passes THEN s2_type WHEN today_live THEN today_type END AS type_v2,
         CASE WHEN passes THEN s2_confidence END AS confidence,
         CASE WHEN passes THEN concat('student-s2:', model_id) WHEN today_type IS NULL THEN 'no_today_type' ELSE 'fallback:today' END AS decided_by,
         CASE WHEN passes THEN 'student-s2' WHEN today_type IS NULL THEN 'no_today_type' ELSE 'fallback:today' END AS outcome,
         CASE WHEN passes THEN s2_trad WHEN today_live THEN today_trad END AS is_traditional,
         substr(sha2(concat_ws('|', '', s2_input_hash, coalesce(today_type, ''), '', ''), 256), 1, 32) AS input_hash,
         s2_type, s2_confidence, s2_input_hash, today_type
  FROM pp
),
-- stage-0 source lists (student/prod stage0.apply): a work matched by lists of exactly one type is settled by the first
-- such list in the policy's order; a list with both a DOI regex and URL substrings needs both to hit
l1 AS (SELECT * FROM lists WHERE NOT (doi_regex IS NOT NULL AND size(url_substrings) > 0)),
l2 AS (SELECT * FROM lists WHERE doi_regex IS NOT NULL AND size(url_substrings) > 0),
s0m AS (
  SELECT b.work_id, l1.ord, l1.list_id, l1.type FROM b CROSS JOIN l1
  WHERE (instr(b.s0_doi, '/') > 0 AND array_contains(l1.doi_prefixes, split_part(b.s0_doi, '/', 1)))
     OR (l1.doi_regex IS NOT NULL AND b.s0_doi RLIKE l1.doi_regex)
  UNION ALL
  SELECT x.work_id, l1.ord, l1.list_id, l1.type
  FROM (SELECT work_id, explode(s0_names) AS n FROM b WHERE s0_names IS NOT NULL) x CROSS JOIN l1
  WHERE (size(l1.names_exact) > 0 OR l1.names_regex IS NOT NULL)
    AND (array_contains(l1.names_exact, x.n) OR (l1.names_regex IS NOT NULL AND x.n RLIKE l1.names_regex))
  UNION ALL
  SELECT x.work_id, s.ord, s.list_id, s.type
  FROM (SELECT work_id, s0_doi AS doi, explode(s0_urls) AS u FROM b WHERE s0_urls IS NOT NULL) x
  CROSS JOIN (SELECT ord, list_id, type, doi_regex, explode(url_substrings) AS sub FROM l2) s
  WHERE x.doi RLIKE s.doi_regex AND instr(x.u, s.sub) > 0
),
s0 AS (
  SELECT work_id, min_by(list_id, ord) AS list_id, min(type) AS s0_type
  FROM (SELECT DISTINCT work_id, ord, list_id, type FROM s0m) m
  GROUP BY work_id
  HAVING count(DISTINCT type) = 1
),
dec AS (
  SELECT b.work_id, b.build_type, b.pl_prov, b.pl_nid, b.policy_id, b.model_id,
         CASE WHEN t0.type IS NOT NULL THEN s0.s0_type ELSE b.type_v2 END AS type_v2,
         CASE WHEN t0.type IS NOT NULL THEN NULL ELSE b.confidence END AS confidence,
         CASE WHEN t0.type IS NOT NULL THEN concat('stage0:', s0.list_id) ELSE b.decided_by END AS decided_by,
         CASE WHEN t0.type IS NOT NULL THEN 'stage0-list' ELSE b.outcome END AS outcome,
         CASE WHEN t0.type IS NOT NULL THEN t0.is_traditional ELSE b.is_traditional END AS is_traditional,
         CASE WHEN t0.type IS NOT NULL THEN substr(sha2(concat_ws('|', b.input_hash, s0.list_id), 256), 1, 32) ELSE b.input_hash END AS input_hash,
         b.s2_type, b.s2_confidence, b.s2_input_hash, b.today_type
  FROM b
  LEFT JOIN s0 ON s0.work_id = b.work_id
  LEFT JOIN ty t0 ON t0.type = s0.s0_type                      -- a list type retired since the policy was made never settles
),
prot AS (SELECT DISTINCT provenance, native_id FROM openalex.works.locations_w_types WHERE classified_rule IN ('#1647: physicalobject sample code ahead of title code -> other')),
del AS (SELECT DISTINCT work_id FROM openalex.works.deleted_works),
prev AS (SELECT CAST(NULL AS BIGINT) AS work_id, CAST(NULL AS STRING) AS pub_type, CAST(NULL AS STRUCT<type: STRING, outcome: STRING, decided_by: STRING, confidence: DECIMAL(5,4), input_hash: STRING, s2_type: STRING, s2_confidence: DECIMAL(5,4), s2_input_hash: STRING, today_type: STRING, policy_id: STRING, model_version: STRING, vocab_version: STRING, is_traditional: BOOLEAN, flip: STRING, served: BOOLEAN, held: BOOLEAN, held_type: STRING, decided_on: DATE>) AS pv WHERE 1 = 0),
-- why the decision differs from last night's published one (null: it doesn't); 'unexplained' = nothing it rests on moved
f AS (
  SELECT dec.*, prev.pub_type, vv.vocab_version, prev.pv.type AS p_type, prev.pv.outcome AS p_outcome,
         prev.pv.decided_by AS p_decided_by, prev.pv.confidence AS p_confidence, prev.pv.is_traditional AS p_trad,
         CASE WHEN prev.pv IS NULL THEN 'new'
              WHEN prev.pv.type IS NOT DISTINCT FROM dec.type_v2 THEN NULL
              WHEN prev.pv.policy_id IS DISTINCT FROM dec.policy_id THEN 'policy'
              WHEN prev.pv.model_version IS DISTINCT FROM dec.model_id THEN 'model'
              WHEN prev.pv.vocab_version IS DISTINCT FROM vv.vocab_version THEN 'vocabulary'
              WHEN prev.pv.s2_input_hash IS DISTINCT FROM dec.s2_input_hash THEN 'input'
              WHEN (prev.pv.outcome = 'stage0-list' OR dec.outcome = 'stage0-list')
                   AND prev.pv.decided_by IS DISTINCT FROM dec.decided_by THEN 'stage0-list'
              WHEN dec.outcome <> 'student-s2' AND prev.pv.today_type IS DISTINCT FROM dec.today_type THEN 'today_type'
              ELSE 'unexplained' END AS flip,
         pr.native_id IS NOT NULL AS protected, dl.work_id IS NOT NULL AS deleted
  FROM dec CROSS JOIN vv
  LEFT JOIN prev ON prev.work_id = dec.work_id
  LEFT JOIN prot pr ON pr.provenance = dec.pl_prov AND pr.native_id = dec.pl_nid
  LEFT JOIN del dl ON dl.work_id = dec.work_id
),
-- an unexplained flip holds last night's decision (type, outcome, ...) until something it rests on moves: the next
-- night compares against the held decision, so the flip stays 'unexplained' (and Check_Flips loud) every night; the
-- new decision waits in held_type (Codex 11-09 #2)
h AS (
  SELECT f.*, flip = 'unexplained' AS held,
         CASE WHEN flip = 'unexplained' THEN p_type ELSE type_v2 END AS d_type,
         CASE WHEN flip = 'unexplained' THEN p_outcome ELSE outcome END AS d_outcome,
         CASE WHEN flip = 'unexplained' THEN p_decided_by ELSE decided_by END AS d_decided_by,
         CASE WHEN flip = 'unexplained' THEN p_confidence ELSE confidence END AS d_confidence,
         CASE WHEN flip = 'unexplained' THEN p_trad ELSE is_traditional END AS d_trad,
         CASE WHEN flip = 'unexplained' THEN type_v2 END AS held_type
  FROM f
),
-- served tonight: rule 1 stage-2 and list decisions only (fallback works keep tonight's cascade type); 2 live types only
-- (both kinds are live by construction); 3 never a removed work; 4 no study-registration written, and Jason's
-- study-registration rows stay; 4b protected cascade rules stay; 5 parent/child (#2065, not live yet); 6 the ramp
sv AS (
  SELECT h.*,
         sw.served = h.policy_id AND h.d_outcome IN ('student-s2', 'stage0-list') AND h.d_type IS NOT NULL
         AND h.d_type <> 'study-registration' AND coalesce(h.build_type, '') <> 'study-registration'
         AND NOT h.protected AND NOT h.deleted
         AND pmod(xxhash64(h.work_id), sw.n_buckets) < sw.k AS serve
  FROM h CROSS JOIN sw
)
SELECT work_id, d_type AS serve_type, CAST(named_struct('type', d_type, 'outcome', d_outcome, 'decided_by', d_decided_by, 'confidence', d_confidence, 'input_hash', input_hash, 's2_type', s2_type, 's2_confidence', s2_confidence, 's2_input_hash', s2_input_hash, 'today_type', today_type, 'policy_id', policy_id, 'model_version', model_id, 'vocab_version', vocab_version, 'is_traditional', d_trad, 'flip', flip, 'served', serve, 'held', held, 'held_type', held_type, 'decided_on', current_date()) AS STRUCT<type: STRING, outcome: STRING, decided_by: STRING, confidence: DECIMAL(5,4), input_hash: STRING, s2_type: STRING, s2_confidence: DECIMAL(5,4), s2_input_hash: STRING, today_type: STRING, policy_id: STRING, model_version: STRING, vocab_version: STRING, is_traditional: BOOLEAN, flip: STRING, served: BOOLEAN, held: BOOLEAN, held_type: STRING, decided_on: DATE>) AS v2
FROM sv
    ) AS s
    ON t.id = s.work_id
    WHEN MATCHED THEN UPDATE SET t.type_decision = s.v2,
      t.type = CASE WHEN s.v2.served THEN s.serve_type ELSE t.type END;
  END IF;
END

### Merge citations and referenced_works

In [ ]:
-- Combined MERGE for referenced_works: the legacy graph (openalex.mid.citation) and the parsed graph
-- (openalex.works.referenced_works), pre-unioned and deduped via COLLECT_SET.
-- oxjob #1256: a cited id that was merged away redirects to its winner (openalex.works.merged_work_ids).
-- oxjob #835: so does the citing id, so a merged-away work's own references move to its winner instead of matching no
-- row (7.64M citations missing from 1.07M winners on 2026-10-26). Both sides use the latest winner, through the one map
-- (as in the cell below). The cell below drops every self-reference.
-- redirect = merged_work_ids_map, built tonight by the Merged_Work_Ids_Map task (oxjob #2006; rules in notebooks/elastic/sync_merged_ids)
WITH redirect AS (
  SELECT id AS k, merge_into_id AS v FROM openalex.works.merged_work_ids_map
),
edges AS (
  SELECT paper_id AS citing, paper_reference_id AS cited FROM openalex.mid.citation
  UNION ALL
  SELECT citing_work_id, EXPLODE(referenced_works) FROM openalex.works.referenced_works
),
combined AS (
  SELECT COALESCE(rc.v, e.citing) AS id, collect_set(COALESCE(rr.v, e.cited)) AS referenced_works
  FROM edges e
  LEFT JOIN redirect rc ON rc.k = e.citing
  LEFT JOIN redirect rr ON rr.k = e.cited
  GROUP BY 1
)
MERGE INTO openalex.works.openalex_works_build AS target
USING combined AS source
ON target.id = source.id
WHEN MATCHED THEN UPDATE SET
  target.referenced_works = slice(ARRAY_SORT(array_union(target.referenced_works, source.referenced_works)), 1, 5000),
  target.referenced_works_count = least(size(array_union(target.referenced_works, source.referenced_works)), 5000)

In [ ]:
-- oxjob #1643: merged-away ids inherited from openalex_works_base. CreateWorksBase copies locations_mapped's referenced_works
-- with no redirect and the MERGE above only unions into them, so a loser id stayed listed (a 404) beside its winner: 23.1M refs
-- on 13.9M works on 2026-10-13. Rewrite exactly the arrays holding a merged-away id or the work's own id: the id becomes its
-- live winner (merged_work_ids_map), and a work never cites itself (oxjob #835, #604). Runs before the citations MERGE so
-- cited_by_count follows.
-- redirect = merged_work_ids_map, built tonight by the Merged_Work_Ids_Map task (oxjob #2006; rules in notebooks/elastic/sync_merged_ids)
WITH redirect AS (
  SELECT id AS k, merge_into_id AS v FROM openalex.works.merged_work_ids_map
),
edges AS (
  SELECT id, explode(referenced_works) AS ref FROM openalex.works.openalex_works_build WHERE size(referenced_works) > 0
),
hit AS (
  SELECT DISTINCT e.id FROM edges e LEFT JOIN redirect r ON r.k = e.ref WHERE r.k IS NOT NULL OR e.ref = e.id
),
fixed AS (
  SELECT e.id, array_sort(collect_set(CASE WHEN COALESCE(r.v, e.ref) <> e.id THEN COALESCE(r.v, e.ref) END)) AS refs
  FROM edges e JOIN hit h ON h.id = e.id LEFT JOIN redirect r ON r.k = e.ref
  GROUP BY e.id
)
MERGE INTO openalex.works.openalex_works_build AS target
USING fixed AS source
ON target.id = source.id
WHEN MATCHED THEN UPDATE SET
  target.referenced_works = slice(source.refs, 1, 5000),
  target.referenced_works_count = least(size(source.refs), 5000)

#### Merge cited_by_count and counts_by_year

In [0]:
-- Calculate and MERGE the citations
-- Far fewer changes than propagating through locations_mapped and 17 CTEs, no need to select distinct work_id data
-- runtime about 1 min, updates 67M rows
WITH exploded_references AS (
  SELECT
    id,
    publication_year,
    EXPLODE(referenced_works) AS cited_work_id
  FROM openalex.works.openalex_works_build
  WHERE referenced_works_count > 0
    AND publication_year <= YEAR(CURRENT_DATE())
    AND type != 'dataset'
),
citation_counts AS (
  SELECT
    cited_work_id,
    publication_year,
    COUNT(*) AS cited_by_count
  FROM exploded_references
  GROUP BY cited_work_id, publication_year
),
citation_counts_by_work AS (
  SELECT 
    cited_work_id,
    FILTER(
      SORT_ARRAY(
        COLLECT_LIST(
          NAMED_STRUCT(
            'year', publication_year,
            'cited_by_count', cited_by_count
          )
        ),
        false
      ),
      x -> x.year >= 2012
    ) AS counts_by_year,
    SUM(cited_by_count) AS cited_by_count -- total across all years
  FROM citation_counts
  GROUP BY cited_work_id
)
MERGE INTO openalex.works.openalex_works_build AS target
USING citation_counts_by_work AS source
ON target.id = source.cited_work_id
WHEN MATCHED THEN
UPDATE SET
  target.cited_by_count = source.cited_by_count,
  target.counts_by_year = source.counts_by_year;



### Merge Concepts, Keywords, and Topics

In [ ]:
-- Combined merge for concepts, keywords, and topics.
-- Keywords come ONLY from openalex.works.work_keywords_v2 (oxjob #1322, student tagger); the concept
-- tagger's work_concepts.keywords is no longer served. Works without a v2 row keep the base [].
-- Topics come ONLY from openalex.works.work_topics_v2 (oxjob #1531, q8b_2m student: top 3, score = the model's
-- probability, [] = not classifiable); work_topics (old BERT + mid backfill) is no longer read and is kept for
-- rollback. Works without a v2 row keep the base [] / NULL. GET, not [0]: [0] on [] raises INVALID_ARRAY_INDEX.
MERGE INTO openalex.works.openalex_works_build AS target
USING (
  SELECT
    COALESCE(wc.work_id, wt.work_id, wk.work_id) AS work_id,
    wc.concepts,
    wk.keywords,
    wt.topics
  FROM openalex.works.work_concepts wc
  FULL OUTER JOIN openalex.works.work_topics_v2 wt
    ON wc.work_id = wt.work_id
  FULL OUTER JOIN openalex.works.work_keywords_v2 wk
    ON wk.work_id = COALESCE(wc.work_id, wt.work_id)
) AS source
ON target.id = source.work_id
WHEN MATCHED THEN UPDATE SET
  target.concepts = COALESCE(source.concepts, target.concepts),
  target.keywords = COALESCE(source.keywords, target.keywords),
  target.topics = COALESCE(source.topics, target.topics),
  target.primary_topic = COALESCE(GET(source.topics, 0), target.primary_topic);

### Merge FWCI, citation percentiles, and cited_by_percentile_year

In [ ]:
-- FWCI + cohort percentile (pub+3 within pub_year/subfield_id/work_type) -- 257,176,397 updated
-- + cited_by_percentile_year (global by eval_year)
-- Computes everything from citation edges (referenced_works); no counts_by_year usage.

WITH base AS (  -- candidate works + work_type mapping
  SELECT
    id AS work_id,
    CASE
      WHEN type IN ('article', 'conference-paper', 'book', 'review', 'book-chapter') THEN type
      ELSE NULL
    END AS work_type,
    COALESCE(publication_year, YEAR(publication_date)) AS pub_year,
    primary_topic.subfield.id AS subfield_id
  FROM openalex.works.openalex_works_build
  WHERE primary_topic.subfield.id IS NOT NULL
    AND COALESCE(publication_year, YEAR(publication_date)) IS NOT NULL
),

-- All citation edges: (citing_year -> cited_work_id)
edges AS (
  SELECT
    w.publication_year AS citing_year,
    EXPLODE(COALESCE(w.referenced_works, ARRAY())) AS cited_work_id
  FROM openalex.works.openalex_works_build AS w
  WHERE w.referenced_works_count > 0
    AND w.publication_year IS NOT NULL
    AND w.publication_year <= YEAR(CURRENT_DATE())
),

-- Per-work pub+3 citations via edges (join + conditional sum)
three_years AS (
  SELECT
    b.work_id,
    b.subfield_id,
    b.pub_year,
    b.work_type,
    SUM(
      CASE
        WHEN e.citing_year BETWEEN b.pub_year AND LEAST(b.pub_year + 3, YEAR(CURRENT_DATE()))
        THEN 1 ELSE 0
      END
    ) AS pub_plus_3_citations
  FROM base b
  LEFT JOIN edges e
    ON e.cited_work_id = b.work_id
  WHERE b.work_type IS NOT NULL
  GROUP BY b.work_id, b.subfield_id, b.pub_year, b.work_type
),

-- Compute FWCI inline (expected citations = cohort average)
with_fwci AS (
  SELECT
    t.work_id,
    t.subfield_id,
    t.pub_year,
    t.work_type,
    t.pub_plus_3_citations,
    CASE
      WHEN AVG(t.pub_plus_3_citations) OVER (
        PARTITION BY t.pub_year, t.subfield_id, t.work_type
      ) <= 0 THEN NULL
      ELSE t.pub_plus_3_citations / AVG(t.pub_plus_3_citations) OVER (
        PARTITION BY t.pub_year, t.subfield_id, t.work_type
      )
    END AS fwci
  FROM three_years t
),

-- Cohort percentile for pub+3 within (pub_year, subfield_id, work_type) + top-1/10 flags
-- Note: is_in_top_1_percent and is_in_top_10_percent are derived directly from citation_pct_cohort
-- to ensure they always match the displayed percentile value (fixes inconsistency bug)
-- oxjob #1271 (9 Nov 2026): works with the same pub+3 count share one value, the share of their cohort with strictly
-- fewer pub+3 citations (PERCENT_RANK's own tie rule). The old work_id tie-break spread each tied block by id, so in
-- cohorts where most works are uncited, uncited works with high ids read top 10% / top 1% (1.69M core works, 9 Nov).
-- A work with 0 pub+3 citations is never flagged: it reads 0 by the tie rule, and the guard states it.
with_percentile AS (
  SELECT
    work_id,
    subfield_id,
    pub_year,
    work_type,
    pub_plus_3_citations,
    ROUND(fwci, 4) AS fwci,
    ROUND(
      PERCENT_RANK() OVER (
        PARTITION BY pub_year, subfield_id, work_type
        ORDER BY pub_plus_3_citations
      ), 8
    ) AS citation_pct_cohort,
    (PERCENT_RANK() OVER (
      PARTITION BY pub_year, subfield_id, work_type
      ORDER BY pub_plus_3_citations
    ) >= 0.99 AND pub_plus_3_citations > 0) AS is_in_top_1_percent,
    (PERCENT_RANK() OVER (
      PARTITION BY pub_year, subfield_id, work_type
      ORDER BY pub_plus_3_citations
    ) >= 0.90 AND pub_plus_3_citations > 0) AS is_in_top_10_percent
  FROM with_fwci
),

/* ===== cited_by_percentile_year (global by eval_year), computed from edges ===== */
/* ===== cited_by_percentile_year (global, min/max across years)
   Build a proper year-level distribution that includes zeros and preserves frequencies ===== */

/* All candidate works and their pub_year (global, not filtered by work_type/subfield) */
-- All works with their pub_year
all_works AS (
  SELECT id AS work_id,
         coalesce(publication_year, year(publication_date)) AS pub_year
  FROM openalex.works.openalex_works_build
  WHERE coalesce(publication_year, year(publication_date)) IS NOT NULL
),

-- Years universe
years AS (
  SELECT explode(sequence(1920, year(current_date()))) AS year
),

-- Total alive works per year (eligible to receive citations)
alive_per_year AS (
  SELECT y.year,
         count(*) AS alive_works
  FROM years y
  JOIN all_works w
    ON y.year >= w.pub_year
  GROUP BY y.year
),

-- Non-zero citation buckets from counts_by_year (use existing precomputed counts)
nonzero_year_freq AS (
  SELECT
    cy.year,
    cy.cited_by_count AS citation_count,
    count(*) AS freq
  FROM (
    SELECT
      explode(coalesce(w.counts_by_year, array())) AS cy
    FROM openalex.works.openalex_works_build w
  )
  WHERE cy.year BETWEEN 1920 AND year(current_date())
    AND cy.cited_by_count > 0
  GROUP BY cy.year, cy.cited_by_count
),

-- Sum of non-zero frequencies per year
nonzero_sum AS (
  SELECT year, sum(freq) AS nonzero_total
  FROM nonzero_year_freq
  GROUP BY year
),

-- Zero bucket derived as alive - nonzero_total
zero_bucket AS (
  SELECT a.year,
         0 AS citation_count,
         greatest(a.alive_works - coalesce(n.nonzero_total, 0), 0) AS freq
  FROM alive_per_year a
  LEFT JOIN nonzero_sum n USING (year)
),

-- Full frequency table including zero bucket
year_count_freq AS (
  SELECT * FROM nonzero_year_freq
  UNION ALL
  SELECT * FROM zero_bucket
),

-- Cumulative distribution per year
year_count_cume AS (
  SELECT
    year,
    citation_count,
    freq,
    sum(freq) OVER (PARTITION BY year) AS total_freq,
    sum(freq) OVER (PARTITION BY year ORDER BY citation_count
                    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cum_freq_inclusive
  FROM year_count_freq
),

-- Bounds for each (year, count)
year_count_bounds AS (
  SELECT
    year,
    citation_count,
    case when total_freq <= 1 then 0.0
         else (cum_freq_inclusive - freq) / total_freq end AS lower_pct,
    case when total_freq = 0 then 0.0
         else cum_freq_inclusive / total_freq end AS upper_pct
  FROM year_count_cume
),

-- Map each work's counts_by_year to bounds
work_year_bands AS (
  SELECT
    w.id AS work_id,
    cy.year,
    b.lower_pct,
    b.upper_pct
  FROM (
    SELECT
      w.id,
      explode(coalesce(w.counts_by_year, array())) AS cy
    FROM openalex.works.openalex_works_build w
  ) w
  JOIN year_count_bounds b
    ON b.year = cy.year AND b.citation_count = cy.cited_by_count
),

-- Collapse to min/max across years per work
year_pct_minmax AS (
  SELECT
    work_id,
    min(lower_pct) AS lower_pct_min,
    max(upper_pct) AS upper_pct_max
  FROM work_year_bands
  GROUP BY work_id
),

formatted_year_pct AS (
  SELECT
    work_id,
    named_struct(
      'min',
        case
          when round(coalesce(lower_pct_min,0)*100)=100 then 99
          when round(coalesce(lower_pct_min,0)*100)=round(coalesce(upper_pct_max,0)*100)
            then greatest(cast(round(coalesce(lower_pct_min,0)*100) as int)-1, 0)
          else cast(round(coalesce(lower_pct_min,0)*100) as int)
        end,
      'max',
        case
          when round(coalesce(upper_pct_max,0)*100)=100 then 100
          else cast(round(coalesce(upper_pct_max,0)*100) as int)
        end
    ) AS cited_by_percentile_year
  FROM year_pct_minmax
),

updates AS (
  SELECT
    p.work_id,
    p.fwci,
    NAMED_STRUCT(
      'value', p.citation_pct_cohort,
      'is_in_top_1_percent', p.is_in_top_1_percent,
      'is_in_top_10_percent', p.is_in_top_10_percent
    ) AS citation_normalized_percentile,
    y.cited_by_percentile_year
  FROM with_percentile p
  LEFT JOIN formatted_year_pct y
    ON y.work_id = p.work_id
)
-- Preview:
-- SELECT * FROM updates;

MERGE INTO openalex.works.openalex_works_build AS target
USING updates AS source
  ON target.id = source.work_id
WHEN MATCHED
THEN UPDATE SET
  target.fwci = source.fwci,
  target.citation_normalized_percentile =
    COALESCE(source.citation_normalized_percentile, target.citation_normalized_percentile),
  target.cited_by_percentile_year =
    COALESCE(source.cited_by_percentile_year, target.cited_by_percentile_year);

### Merge `sustainable_development_goals` (Jev-trained head, oxjob #1300)

In [ ]:
-- sustainable_development_goals = the Jev-trained SDG head (oxjob #1300; Jason 2026-10-01: full replacement of
-- Aurora, no shadow field). Source openalex.works.works_sdg_jev, written by notebooks/sdg/sdg_jev_head_score
-- (mode=full once, then mode=incremental as task sdg_score of Embed_Qwen3_Nightly as new vectors land).
-- Goals at the head's threshold (0.4), {id, display_name, score}, score = calibrated score (the classifier's
-- confidence), highest first. A work with no head row (no vector yet) or no goal over threshold keeps the base's
-- empty list: never Aurora. No skip-if-missing on purpose: without works_sdg_jev this fails loudly instead of
-- silently emptying the field on ~180M works. In the content hash (hash cell below).
MERGE INTO openalex.works.openalex_works_build AS target
USING (
  SELECT work_id, sdgs AS sustainable_development_goals
  FROM openalex.works.works_sdg_jev
  WHERE SIZE(sdgs) > 0
) AS source
ON target.id = source.work_id
WHEN MATCHED THEN UPDATE SET
  target.sustainable_development_goals = source.sustainable_development_goals;

### Merge `study_designs` (oxjob #1312)

In [ ]:
-- Study design per work from the Study Design job (jobs/study_design.yaml, served table rebuilt nightly
-- 19:00 UTC): automated tagging wherever it ran, else PubMed's MEDLINE tags (2026-09-26); seven PubMed values only.
-- Slugs and display names must match utils/study_design.py (VALUE_ID / SERVED_CLASSES) and the
-- study-designs entity (notebooks/elastic/sync_study_designs.py). Canonical order is kept from the source array.
MERGE INTO openalex.works.openalex_works_build AS target
USING (
  SELECT work_id,
    TRANSFORM(FILTER(study_designs, s -> s IN ('randomized-controlled-trial', 'clinical-trial', 'observational-study',
                                            'case-report', 'systematic-review', 'meta-analysis', 'study-protocol')),
      s -> NAMED_STRUCT(
      'id', CONCAT('https://openalex.org/study-designs/', s),
      'display_name', CASE s
        WHEN 'randomized-controlled-trial' THEN 'Randomized Controlled Trial'
        WHEN 'clinical-trial' THEN 'Clinical Trial'
        WHEN 'observational-study' THEN 'Observational Study'
        WHEN 'case-report' THEN 'Case Report'
        WHEN 'systematic-review' THEN 'Systematic Review'
        WHEN 'meta-analysis' THEN 'Meta-Analysis'
        WHEN 'study-protocol' THEN 'Study Protocol'
      END)) AS study_designs
  FROM openalex.works.works_study_design
  WHERE SIZE(study_designs) > 0
) AS source
ON target.id = source.work_id
WHEN MATCHED THEN UPDATE SET
  target.study_designs = source.study_designs;

### Merge `awards`

In [0]:
MERGE INTO openalex.works.openalex_works_build AS target
USING (
  SELECT work_id,
    ARRAY_SORT(
      collect_set(award),
      (left, right) -> CASE
        WHEN left.id < right.id THEN -1
        WHEN left.id > right.id THEN 1
        ELSE 0
      END
    ) as awards
  FROM openalex.awards.work_awards
  WHERE work_id IS NOT NULL
  GROUP BY work_id
) as source
ON target.id = source.work_id
WHEN MATCHED THEN UPDATE SET
  target.awards = source.awards;

### Merge `funders`

In [ ]:
-- Work redirects for funder enrichment (cell below): the one map, merged_work_ids_map, built tonight by the
-- Merged_Work_Ids_Map task (oxjob #2006; rules in notebooks/elastic/sync_merged_ids). A loser that is live again is not
-- in it, so it keeps its own funders; an id the map can't resolve stays unchanged (the COALESCE below).
CREATE OR REPLACE TABLE openalex.works.funder_work_survivors USING DELTA AS
SELECT id AS loser_work_id, merge_into_id AS survivor_work_id FROM openalex.works.merged_work_ids_map



In [0]:
%sql
-- 1) Build rolled funders from backfill + FULLTEXT + GTR + Crossref
WITH funder_projection AS (
  -- openalex.funders.funders is the authority for both merge state and current metadata.
  -- One row per source id; aliases project the winner's id/ROR/display name.
  SELECT
    source.funder_id AS source_funder_id,
    COALESCE(source.merge_into_id, source.funder_id) AS canonical_funder_id,
    canonical.ror_id AS ror,
    canonical.display_name AS display_name
  FROM openalex.funders.funders source
  LEFT JOIN openalex.funders.funders canonical
    ON canonical.funder_id = COALESCE(source.merge_into_id, source.funder_id)
),
from_backfill AS (
  SELECT
    paper_id AS work_id,
    funder_id
  FROM openalex.mid.work_funder
),
from_backfill_enriched AS (
  SELECT
    b.work_id,
    CONCAT("https://openalex.org/F", COALESCE(fp.canonical_funder_id, b.funder_id)) AS funder_id,
    fp.ror,
    fp.display_name
  FROM from_backfill b
  LEFT JOIN funder_projection fp
    ON fp.source_funder_id = b.funder_id
),
from_fulltext_enriched AS (
  SELECT
    ft.work_id,
    COALESCE(CONCAT("https://openalex.org/F", fp.canonical_funder_id), ft.funder_id) AS funder_id,
    fp.ror,
    COALESCE(fp.display_name, ft.funder_display_name) AS display_name
  FROM openalex.works.fulltext_work_funders ft
  JOIN openalex.common.funder_names_keep keep ON keep.name = ft.funder_name
  LEFT JOIN funder_projection fp
    ON CONCAT("https://openalex.org/F", fp.source_funder_id) = ft.funder_id
),
from_gtr AS (
  SELECT
    g.work_id,
    COALESCE(CONCAT("https://openalex.org/F", fp.canonical_funder_id), g.funder.id) AS funder_id,
    fp.ror,
    COALESCE(fp.display_name, g.funder.display_name) AS display_name
  FROM openalex.awards.gtr_awards g
  LEFT JOIN funder_projection fp
    ON CONCAT("https://openalex.org/F", fp.source_funder_id) = g.funder.id
  WHERE g.work_id IS NOT NULL
),
from_crossref AS (
  SELECT
    cwf.work_id,
    CONCAT("https://openalex.org/F", COALESCE(fp.canonical_funder_id, cwf.funder_id)) AS funder_id,
    fp.ror,
    fp.display_name
  FROM openalex.awards.crossref_work_funders cwf
  LEFT JOIN funder_projection fp ON fp.source_funder_id = cwf.funder_id
),
from_work_awards AS (
  -- oxjobs #371: derive funders straight from the unified work_awards so .funders covers the
  -- funder of EVERY award (.funders >= funders(.awards)) for all feeds (NWO + future) with no
  -- per-feed leg. Known funders only (award strings with a null funder are exempt, kept as-is).
  SELECT
    wa.work_id,
    COALESCE(CONCAT("https://openalex.org/F", fp.canonical_funder_id), wa.award.funder_id) AS funder_id,
    fp.ror,
    COALESCE(fp.display_name, wa.award.funder_display_name) AS display_name
  -- ce1014: a link taken out by a pinned SUPPRESS list (Nightly Grants list_legs.py) keeps its funder on the work
  FROM (SELECT work_id, award FROM openalex.awards.work_awards
        UNION ALL SELECT served_work_id, award FROM openalex.awards.work_award_list_audit
        WHERE action = 'SUPPRESS' AND status = 'REMOVED') wa
  LEFT JOIN funder_projection fp
    ON CONCAT("https://openalex.org/F", fp.source_funder_id) = wa.award.funder_id
  WHERE wa.award.funder_id IS NOT NULL
),
from_funder_reported AS (
  -- OUTPUT-LIST pattern: funder-asserted work->funder edges from funders that
  -- publish their funded-publications list (no grants). Mirrors from_crossref;
  -- reads the shared funder_reported_work_funders union table.
  SELECT
    fr.work_id,
    CONCAT("https://openalex.org/F", COALESCE(fp.canonical_funder_id, fr.funder_id)) AS funder_id,
    fp.ror,
    fp.display_name
  FROM openalex.awards.funder_reported_work_funders fr
  LEFT JOIN funder_projection fp ON fp.source_funder_id = fr.funder_id
),
unioned AS (
  SELECT work_id, funder_id, ror, display_name FROM from_backfill_enriched
  UNION ALL  
  SELECT work_id, funder_id, ror, display_name FROM from_fulltext_enriched
  UNION ALL
  SELECT work_id, funder_id, ror, display_name FROM from_gtr
  UNION ALL
  SELECT work_id, funder_id, ror, display_name FROM from_crossref
  UNION ALL
  SELECT work_id, funder_id, ror, display_name FROM from_work_awards
  UNION ALL
  SELECT work_id, funder_id, ror, display_name FROM from_funder_reported
),
work_projected AS (
  -- Work redirects are independent of the funder-identity projection above.
  -- Resolve ALL legs before dedup, including already redirected award links.
  SELECT COALESCE(m.survivor_work_id, u.work_id) AS work_id,
         u.funder_id, u.ror, u.display_name
  FROM unioned u
  LEFT JOIN openalex.works.funder_work_survivors m ON m.loser_work_id = u.work_id
  UNION ALL
  -- Keep original pairs while a redirected loser is still a live target row.
  SELECT u.work_id, u.funder_id, u.ror, u.display_name
  FROM unioned u
  JOIN openalex.works.funder_work_survivors m ON m.loser_work_id = u.work_id
  JOIN openalex.works.openalex_works_build live ON live.id = u.work_id
),
withdrawn_funders AS (
  -- Funders withdrawn from the registry (openalex.funders.deleted_funders ledger: the empty-funder batches and
  -- dd1016-b3, no ROR and no award, Kyle 10-08/10-09). Their work links are removed, not redirected: no leg may
  -- serve them even if a source table still carries the id.
  SELECT DISTINCT CONCAT("https://openalex.org/F", funder_id) AS funder_id FROM openalex.funders.deleted_funders
),
dedup AS (
  -- Work and funder canonical ids collapse duplicate edges. Known ids carry the same
  -- live values on every leg, so MAX is only a deterministic tie-break.
  -- Unknown ids are retained by the LEFT JOIN fallbacks above with ror=NULL.
  SELECT
    work_id,
    funder_id,
    MAX(display_name) AS display_name,
    MAX(ror) AS ror
  FROM work_projected
  LEFT ANTI JOIN withdrawn_funders x ON x.funder_id = work_projected.funder_id
  GROUP BY work_id, funder_id
),
rolled_up AS (
  SELECT
    work_id,
    -- order by funder_id via lexicographic struct ordering (id is first field)
    sort_array(
      collect_list(
        struct(
          funder_id as id,
          display_name,
          ror
        )
      )
    ) AS funders
  FROM dedup
  GROUP BY work_id
),
withdrawn_hits AS (
  -- Served (work, funder) entries that point at a withdrawn funder.
  SELECT e.work_id, e.funder
  FROM (SELECT w.id AS work_id, explode(w.funders) AS funder FROM openalex.works.openalex_works_build w) e
  LEFT SEMI JOIN withdrawn_funders x ON x.funder_id = e.funder.id
),
withdrawn_only AS (
  -- A work whose every source link went with a withdrawn funder has no rolled_up row, so the MERGE below would
  -- leave its old array in place. Strip only the withdrawn entries; anything else it carries is kept as is.
  SELECT w.id AS work_id, sort_array(array_except(w.funders, h.funders)) AS funders
  FROM openalex.works.openalex_works_build w
  JOIN (SELECT work_id, collect_list(funder) AS funders FROM withdrawn_hits GROUP BY work_id) h ON h.work_id = w.id
  LEFT ANTI JOIN rolled_up r ON r.work_id = w.id
)
-- 2) Merge into openalex_works.funders (array<struct<id:string, ror:string, display_name:string>>)
MERGE INTO openalex.works.openalex_works_build AS target
USING (SELECT work_id, funders FROM rolled_up UNION ALL SELECT work_id, funders FROM withdrawn_only) AS source
  ON target.id = source.work_id
WHEN MATCHED THEN
  UPDATE SET target.funders = source.funders;

### Merge `authorships` and institution/country counts

In [0]:
MERGE INTO openalex.works.openalex_works_build AS target
USING openalex.works.work_authorships AS source
ON target.id = source.work_id
WHEN MATCHED THEN UPDATE SET
  target.authorships = slice(source.authorships, 1, 5000),
  target.authors_count = least(source.authors_count, 5000),
  target.corresponding_author_ids = source.corresponding_author_ids,
  target.corresponding_institution_ids = source.corresponding_institution_ids,
  target.institutions_distinct_count = source.institutions_distinct_count,
  target.countries_distinct_count = source.countries_distinct_count;

### Merge `related_works` and full-text

In [0]:
-------- Merge related_works + fulltext --------
-- Runs last so earlier merges never rewrite fulltext-carrying rows;
-- must stay before the hash cell below.
MERGE INTO openalex.works.openalex_works_build AS target
USING (
    SELECT
        COALESCE(r.work_id, f.work_id) AS work_id,
        r.related_works,
        f.fulltext
    FROM (
        -- oxjob #1648: kNN on the Qwen3 vectors (policy v1.2) for every titled work; was related_works_backfill
        -- (a 2025-09-15 concept-based copy, kept for rollback). The switch rode a per-row content-hash rebaseline,
        -- so it re-stamped nothing; revert = point this back at related_works_backfill AND run
        -- oxjobs #1648 scratch/write/write_path.py rebaseline-revert before the same run.
        SELECT work_id, related_works
        FROM openalex.works.work_related_works
        WHERE related_works IS NOT NULL
          AND SIZE(related_works) > 0
    ) r
    FULL OUTER JOIN (
        SELECT work_id, fulltext
        FROM openalex.works.work_fulltext
        WHERE fulltext IS NOT NULL
          AND LENGTH(fulltext) > 0
    ) f ON f.work_id = r.work_id
) AS source
ON target.id = source.work_id
WHEN MATCHED THEN
UPDATE SET
  target.related_works = COALESCE(source.related_works, target.related_works),
  target.fulltext      = COALESCE(source.fulltext, target.fulltext);

In [ ]:
-- Per-run scratch (recreated every run, standard CoR-intermediate pattern):
-- materialize current content hashes once; consumed by the two MERGEs below.
-- Expression must stay byte-identical to the bootstrap in the markdown cell above.
CREATE OR REPLACE TABLE openalex.works.openalex_works_hash_staging AS
SELECT
  id,
  xxhash64(CONCAT_WS('|',
      CAST(id AS STRING),
      COALESCE(doi, ''),
      COALESCE(TO_JSON(ids), '[]'),
      COALESCE(title, ''),
      COALESCE(CAST(publication_date AS STRING), ''),
      COALESCE(CAST(publication_year AS STRING), ''),
      COALESCE(type, ''),
      COALESCE(language, ''),
      COALESCE(abstract, ''),
      COALESCE(TO_JSON(referenced_works), '[]'),
      COALESCE(TO_JSON(TRANSFORM(topics, t -> STRUCT(
        t.id, t.display_name, ROUND(t.score, 4) as score,
        t.subfield, t.field, t.domain
      ))), '[]'),
      COALESCE(TO_JSON(TRANSFORM(concepts, c -> STRUCT(
        c.id, c.wikidata, c.display_name, c.level, ROUND(c.score, 4) as score
      ))), '[]'),
      COALESCE(TO_JSON(TRANSFORM(keywords, k -> STRUCT(
        k.id, k.display_name, ROUND(k.score, 4) as score
      ))), '[]'),
      -- authorships[].author.observed_orcids (oxjob #1340): an array of size <= 1 carries
      -- no information beyond author.orcid (element 0 is always the primary), so it is
      -- stripped before hashing and pre-#1340 rows hash unchanged; only the ~17.9M works
      -- with a multi-ORCID author re-stamp on the first run (the #1267 e242b224 trick).
      -- Must stay identical to the rebaseline/bootstrap expression.
      COALESCE(REGEXP_REPLACE(TO_JSON(authorships), ',?"observed_orcids":\\[("[^"]*")?\\]', ''), '[]'),
      COALESCE(TO_JSON(funders), '[]'),
      -- source.issn: [] ≡ NULL (oxjob #548); location-level "updated" is
      -- bookkeeping, not content — excluded so upstream openalex_updated_dt
      -- bumps don't masquerade as work changes (121M false churn, oxjob #733).
      -- source.listed_in (oxjob #1205) is stripped too: adding it to the
      -- embedded source on ~100M works must not re-stamp them (Casey) — the
      -- attribute trickles into ES/snapshots as works change for other reasons.
      -- location.doi (oxjob #1573) is hashed since oxjob #1616; primary_location and best_oa_location are copies
      -- of a location, so their doi is covered too.
      -- Must stay identical to the rebaseline/bootstrap expression.
      COALESCE(REPLACE(REPLACE(REGEXP_REPLACE(REGEXP_REPLACE(TO_JSON(locations), '"listed_in":\\[[^\\]]*\\],?', ''), '"updated":"[^"]*",?', ''), '"issn":[],', ''), '"issn":[]}', '}'), '[]'),
      COALESCE(TO_JSON(awards), '[]'),
      COALESCE(TO_JSON(STRUCT(open_access.is_oa, open_access.oa_status, open_access.oa_url)), '{}'),
      COALESCE(CAST(is_retracted AS STRING), 'false'),
      COALESCE(publisher, ''),
      COALESCE(TO_JSON(biblio), '{}'),
      COALESCE(TO_JSON(related_works), '[]'),
      COALESCE(TO_JSON(indexed_in), '[]'),
      COALESCE(CAST(indexed_in_crossref AS STRING), 'false'),
      COALESCE(CAST(is_xpac AS STRING), 'false'),
      COALESCE(TO_JSON(mesh), '[]'),
      COALESCE(fulltext, ''),
      COALESCE(abstract_inverted_index, ''),
      -- sustainable_development_goals: the Jev-trained head since oxjob #1300 (was Aurora).
      COALESCE(TO_JSON(TRANSFORM(sustainable_development_goals, s -> STRUCT(
        s.id, s.display_name, ROUND(s.score, 4) as score
      ))), '[]'),
      COALESCE(TO_JSON(corresponding_author_ids), '[]'),
      COALESCE(TO_JSON(corresponding_institution_ids), '[]'),
      COALESCE(CAST(cited_by_count AS STRING), '0'),
      COALESCE(TO_JSON(counts_by_year), '[]'),
      COALESCE(TO_JSON(apc_list), '{}'),
      COALESCE(TO_JSON(apc_paid), '{}'),
      -- study_designs (oxjob #1312): NULL when empty, and CONCAT_WS skips NULLs, so works
      -- without a study design hash exactly as before; only works that carry one re-stamp.
      -- Must stay identical to the rebaseline/bootstrap expression.
      CASE WHEN SIZE(study_designs) > 0 THEN TO_JSON(study_designs) END,
      -- volume (oxjob #1554): NULL when the work is not part of another work, so only works that carry one re-stamp.
      CASE WHEN volume IS NOT NULL THEN TO_JSON(volume) END,
      -- top-10% / top-1% flags (oxjob #1917): a flip now moves updated_date (124,547-297,791 a night did not). NULL for
      -- works in neither, so CONCAT_WS skips it and only the 27.2M flagged works changed hash; #1917 rebaselined them on
      -- 29 Oct 2026 (openalex.transient.oxjob1917_hash_rebase), so none re-stamped for the change itself. fwci and the
      -- percentile value stay out: they move nightly for 17-50% of works. Must stay identical to the rebaseline/bootstrap expression.
      CASE WHEN citation_normalized_percentile.is_in_top_10_percent OR citation_normalized_percentile.is_in_top_1_percent
        THEN TO_JSON(STRUCT(citation_normalized_percentile.is_in_top_10_percent AS top10, citation_normalized_percentile.is_in_top_1_percent AS top1)) END
    )) AS content_hash
FROM openalex.works.openalex_works_build
-- Dedupe guard: duplicate NEW ids would otherwise double-INSERT into the
-- persistent table and permanently break every later MERGE on it.
QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY content_hash) = 1

In [ ]:
-- Maintain the persistent hash table (oxjob #679): updated_date changes ONLY
-- when content_hash changes, and survives every clone/rebuild. Runs BEFORE the
-- publish cell (last), which serves these dates, so there is exactly one timestamp source.
MERGE INTO openalex.works.openalex_works_hash AS target
-- hash_rebaseline (oxjob #1386): on a night with a pre-cleared 'hash_rebaseline' row in openalex.works.e2e_overrides
-- (scripts/preclear_e2e.py), changed works take the new hash but keep their updated_date: the rebaseline for a change
-- over ~100M works, done by the run that computes the new content (ES and Lakebase catch up in id_mod chunks).
-- New works are stamped as always. The flag is a source column because Databricks rejects EXISTS subqueries inside
-- MERGE ... UPDATE SET.
USING (
  SELECT s.*, openalex.works.e2e_override_active('hash_rebaseline', 'false') AS rebaseline
  FROM openalex.works.openalex_works_hash_staging s
) AS source
ON target.id = source.id
WHEN MATCHED AND target.content_hash <> source.content_hash THEN
  UPDATE SET target.content_hash = source.content_hash,
             target.updated_date = CASE WHEN source.rebaseline THEN target.updated_date ELSE CURRENT_TIMESTAMP() END
WHEN NOT MATCHED THEN
  INSERT (id, updated_date, content_hash)
  VALUES (source.id, CURRENT_TIMESTAMP(), source.content_hash)
WHEN NOT MATCHED BY SOURCE THEN DELETE

## `openalex_works_created_date` is persistent (oxjob #1873)

`created_date` is derived in CreateWorksBase as the earliest `openalex_created_dt` over the work's live locations, and
an anchor's date is the day `MapWorkIds` first inserts its `location_work_ids` row. So every path that deletes and
re-inserts anchors (RepointWorkIds, TrackDeletedWorks purges, merges) or replaces a work's oldest records moved the
work's `created_date` later: ~2M works were re-dated Aug-Oct 2026 and showed up as new to `from_created_date` /
`sort=created_date`, the paid "works added since" axis.

This table `(id, created_date)` is the **authoritative owner of `created_date`**, as `openalex_works_hash` is for
`updated_date` (#679), with two differences: a row is never deleted (an id that leaves and comes back keeps its
date), and a stored date only ever moves earlier. The MERGE below adds new ids and lowers a stored date when the
derived one is earlier; the publish cell (last) serves the stored date as `openalex_works.created_date` (#2004). created_date is
not in the content hash, so these cells never re-stamp a work by themselves: a corrected date reaches ES when the
work next re-stamps (to push one, flip its stored `content_hash` before a run).

Bootstrap (one-time, oxjob #1873 `work/sql/08_bootstrap.sql`): seeded with the dates served on 26 Oct 2026.
Revert: delete the two code cells below, and in the publish cell serve `b.created_date` and drop the created_date
MERGE; `openalex_works` then serves the derived date again on the next run.


In [ ]:
-- oxjob #1873: normally a no-op (bootstrapped once). If the table is ever lost it comes back empty and the MERGE below
-- refills it with this run's derived dates.
CREATE TABLE IF NOT EXISTS openalex.works.openalex_works_created_date (
  id BIGINT,
  created_date DATE
)
CLUSTER BY (id)

In [ ]:
-- oxjob #1873: keep the earliest created_date each work id has had. New ids are added, a stored date moves only
-- earlier, rows are never deleted (unlike openalex_works_hash's NOT MATCHED BY SOURCE DELETE).
MERGE INTO openalex.works.openalex_works_created_date AS target
USING (
  SELECT id, MIN(created_date) AS created_date
  FROM openalex.works.openalex_works_build
  WHERE created_date IS NOT NULL
  GROUP BY id
) AS source
ON target.id = source.id
WHEN MATCHED AND source.created_date < target.created_date THEN
  UPDATE SET target.created_date = source.created_date
WHEN NOT MATCHED THEN
  INSERT (id, created_date) VALUES (source.id, source.created_date)

## Publish the finished build (oxjob #2004)

`openalex_works` holds only finished builds. States, one table each:

- `openalex_works_build`: tonight's build. Half-built while this notebook runs; finished when the cell above has run. Nothing outside this notebook reads it.
- `openalex_works`: the last finished build, replaced in one commit by the cell below. Optimize_Works_Enriched compacts it next (no data change) and Guardrails checks it after that.

Before #2004 this notebook cloned `openalex_works_base` straight into `openalex_works` and ran ~20 MERGEs on it in place for 30-60 minutes, and about 20 scheduled jobs read it at fixed UTC times: on 31 Oct the Authors job read it mid-rebuild and the API lost 2.5M ORCID profiles for ~22 hours.

How the cell below publishes. Normal nights: one `INSERT OVERWRITE ... BY NAME` of the build joined to the stored dates (`openalex_works_hash` for `updated_date`, #679; `openalex_works_created_date` for `created_date`, #1873), so the table is written once. Until 5 Nov 2026 the night rewrote it three times: two whole-table date MERGEs, then an 11-minute copy. The INSERT runs only when it gives exactly what the old path gives: `openalex_works` has the build's columns and types (an INSERT can't drop or retype a column, nested fields included) and each date table has one row per id. Otherwise (a night after a schema change, or a duplicate id) the cell runs the old path: the two date MERGEs into the build, then a DEEP CLONE that takes the build's schema (~20 minutes more; a duplicate id fails the MERGE, as it always did).

Revert: `git revert` the #2004 commits (cell 3 clones into `openalex_works` again, the date MERGEs come back and everything runs in place); `openalex_works_build` can then be dropped.

In [ ]:
-- oxjob #2004: publish. One commit replaces openalex_works with the finished build and serves each work's stored
-- dates: updated_date from openalex_works_hash (#679), created_date from openalex_works_created_date (#1873); a NULL
-- or missing stored date keeps the build's.
-- Normal nights: one INSERT OVERWRITE ... BY NAME, the table written once. It runs only when it must give what the
-- old path gives: openalex_works has exactly the build's columns and types (typeof(struct(*)): names, order, nested
-- fields; an INSERT can't drop or retype a column) and each date table has one row per id (a duplicate would
-- duplicate works here; the old MERGE fails on it). Otherwise the old path: both date MERGEs into the build, then a
-- DEEP CLONE that takes the build's schema (~20 min more), with the same table properties as before #2004.
BEGIN
  DECLARE fast BOOLEAN;
  SET fast = coalesce(
        (SELECT typeof(struct(*)) FROM openalex.works.openalex_works LIMIT 1)
          = (SELECT typeof(struct(*)) FROM openalex.works.openalex_works_build LIMIT 1)
    AND (SELECT count(*) = count(DISTINCT id) FROM openalex.works.openalex_works_hash)
    AND (SELECT count(*) = count(DISTINCT id) FROM openalex.works.openalex_works_created_date), FALSE);
  IF fast THEN
    INSERT OVERWRITE openalex.works.openalex_works BY NAME
    SELECT b.* EXCEPT (updated_date, created_date),
           coalesce(h.updated_date, b.updated_date) AS updated_date,
           coalesce(c.created_date, b.created_date) AS created_date
    FROM openalex.works.openalex_works_build b
    LEFT JOIN openalex.works.openalex_works_hash h ON h.id = b.id
    LEFT JOIN openalex.works.openalex_works_created_date c ON c.id = b.id;
  ELSE
    -- Copy authoritative dates from the persistent hash table back into the
    -- column so every consumer keeps reading openalex_works.updated_date as
    -- normal (oxjob #679). Covers both this run's changes (stamped in the hash
    -- table above) and rows whose date the DEEP CLONE reverted to works_base
    -- values. NULL guard: a hash row seeded with a NULL date must never
    -- overwrite a real date.
    MERGE INTO openalex.works.openalex_works_build AS target
    USING openalex.works.openalex_works_hash AS source
    ON target.id = source.id
    WHEN MATCHED AND source.updated_date IS NOT NULL
      AND target.updated_date IS DISTINCT FROM source.updated_date THEN
      UPDATE SET target.updated_date = source.updated_date;
    -- oxjob #1873: serve the stored date. Changes only works whose derived date moved later (anchors deleted and
    -- re-inserted, oldest records gone) or whose stored date was corrected. NULL guard as in the #679 cell above.
    MERGE INTO openalex.works.openalex_works_build AS target
    USING openalex.works.openalex_works_created_date AS source
    ON target.id = source.id
    WHEN MATCHED AND source.created_date IS NOT NULL
      AND target.created_date IS DISTINCT FROM source.created_date THEN
      UPDATE SET target.created_date = source.created_date;
    CREATE OR REPLACE TABLE openalex.works.openalex_works
    DEEP CLONE openalex.works.openalex_works_build
    TBLPROPERTIES (
      'delta.dataSkippingNumIndexedCols' = 36,
      'delta.deletedFileRetentionDuration' = '14 days',
      'delta.logRetentionDuration' = '14 days'
    );
  END IF;
END